# 00 Engine, testbed and verification

Walk-in clinic shift staffing study (Francisco Benita, Tecnologico de Monterrey).

This notebook comes first. It does four things:

1. It writes the core module `clinic_core.py` (the simulation engine, the analytic rules and the analysis helpers) to `clinic2026_paper_v3/core/` on Drive. Every other notebook imports it from there, so the code is in one place.
2. It runs the unit tests of the engine and of the analytic rules, and the Erlang C check of the engine.
3. It checks that the engine reproduces numbers of the earlier version of the study exactly.
4. It writes the parameter table and the Erlang C table of the paper.

**Do not edit the core module cell.** The enumeration of notebook 01 was produced with this module. Running the notebook again writes the same file, so the enumeration stays valid.

**Outputs**

| File | Use in the paper |
|---|---|
| `tables/params.tex` | Table 2, parameters and factor levels |
| `tables/verification.tex` | Table B.1, engine against Erlang C |
| `results/unit_tests.csv`, `results/verification_erlang_c.csv`, `results/timing.csv` | Records |

**Run time:** about 3 minutes when `results/verification_erlang_c.csv` already exists, because the Erlang C check is then read from that file (set `RERUN_ERLANG_C = True` to simulate it again, about 10 minutes).

In [1]:
# Setup: storage folder and numba
import os, sys, shutil, subprocess, time, math
import numpy as np
import pandas as pd

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/My Drive/clinic2026_final'
else:
    BASE = os.environ.get('CLINIC_BASE', os.path.abspath('clinic2026_final'))
os.makedirs(os.path.join(BASE, 'core'), exist_ok=True)
try:
    import numba  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numba'], check=True)
print('Project folder:', BASE)

Mounted at /content/drive
Project folder: /content/drive/My Drive/clinic2026_final


## 1. The core module

The next cell writes `clinic_core.py` to the working directory. The cell after it copies the file to Drive and imports it. Its sections are:

- parameters and factor grid;
- arrival profile and settings;
- common random numbers;
- reference engine and compiled engine;
- evaluator;
- plan geometry and the chunked enumeration queue;
- fronts and structure of the feasible sets;
- stationary baselines and the modified offered load;
- decision-level comparison and error classes;
- descent;
- reliability;
- mechanism and verification;
- output helpers.

In [2]:
%%writefile clinic_core.py
"""clinic_core: simulation engine, analytic baselines and analyses of the walk-in clinic shift-staffing study.

Every notebook of the study imports this single module, so the engine and the analyses cannot drift apart. It is a
port of the version 2 research code restricted to what the paper uses: one last admission time (c = 480, admission
until closing), two and three staffing blocks, exact enumeration, the stationary baselines (SIPP-avg, Lag-SIPP and
SIPP-max under the per-block and daily rules) and the modified offered load (MOL), the structure of the feasible sets,
sampling checks and a first-improvement descent from an analytic plan. The numerical code is unchanged from version 2,
so every number of version 2 at c = 480 is reproduced exactly.

Contents
    1. Parameters and factor grid          9. Stationary baselines (SIPP) and MOL
    2. Arrival profile and settings        10. Decision-level comparison and error classes
    3. Common random numbers               11. Two-dimensional indicators
    4. Reference engine (plain Python)     12. Descent from an analytic plan
    5. Compiled engine (numba)             13. Sampling and reliability
    6. Evaluator                           14. Mechanism and verification
    7. Plan geometry and storage           15. Output helpers (paths, LaTeX tables, figures)
    8. Fronts and structure of the feasible sets
"""
import glob
import heapq
import itertools
import math
import os
import time
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from scipy import stats
from scipy.special import ndtr

try:
    from numba import njit, prange
    HAVE_NUMBA = True
except ImportError:                                    # pure-Python fallback (correct but slow)
    HAVE_NUMBA = False

    def njit(*args, **kwargs):
        if len(args) == 1 and callable(args[0]) and not kwargs:
            return args[0]
        return lambda f: f

    prange = range

# =====================================================================================================================
# 1. Parameters and factor grid
# =====================================================================================================================
T = 480.0                                          # scheduled closing time (minutes after opening)
CUTOFF = 480.0                                     # last admission time used in the paper (admission until closing)
WAGES = np.array([110.0, 160.0, 250.0, 130.0])     # $ per role member per 8-hour day (receptionist .. pharmacist)
ROLE_NAMES = ("Rec", "Nur", "Doc", "Pha")
ROLE_LONG = ("receptionist", "nurse", "physician", "pharmacist")
KAPPA = 1.5                                        # overtime premium on the per-minute wage
C_W = WAGES[1] / T * 0.5                           # waiting cost, $ per patient-minute (half the nurse's wage)
OMEGA = 30.0                                       # standard: total queueing time of at most OMEGA minutes ...
TAU = 0.90                                         # ... for at least TAU of admitted patients, on average over days
THRESHOLDS = np.array([15.0, 20.0, 30.0, 45.0, 60.0])
I_OMEGA = 2                                        # position of OMEGA in THRESHOLDS
ROUTE = np.array([0, 1, 2, 3, 0], dtype=np.int64)  # role used at each of the five stages
STAGE_MEANS = np.array([6.0, 10.0, 15.0, 8.0, 6.0])
MAX_PATIENTS = 400                                 # random numbers drawn per day (far more than any setting brings)
INF = np.inf

MEAN_RATE = 0.10                                   # walk-ins per minute (48 per day)
PARS = (1.0, 1.3, 1.6, 2.0)                        # peak-to-average ratio of the arrival rate
CVS = {"moderate": (0.5, 0.5, 0.5, 0.5, 0.5),      # coefficient of variation of the five stages
       "high": (0.5, 1.0, 1.0, 0.5, 0.5)}
BLOCKS = (2, 3)
LEVELS = 3
MAIN = (200, 0)                                    # main sample: 200 days, seeds 1..200
INDEPENDENT = [(200, 10_000 * k) for k in range(1, 5)]   # four independent samples of 200 days (two blocks)
REFERENCE = (1000, 100_000)                        # reference sample of 1,000 days
VARIANTS = ("avg-block", "avg-daily", "lag-block", "lag-daily", "max-block", "max-daily", "mol-daily")
MAIN_VARIANTS = ("avg-block", "avg-daily", "lag-block", "mol-daily")
VARIANT_LABEL = {"avg-block": "SIPP-avg, per-block rule", "avg-daily": "SIPP-avg, daily rule",
                 "lag-block": "Lag-SIPP, per-block rule", "lag-daily": "Lag-SIPP, daily rule",
                 "max-block": "SIPP-max, per-block rule", "max-daily": "SIPP-max, daily rule", "mol-daily": "MOL"}
THRESHOLD_SETS = {"strict": (0.05, 2.5, 0.025), "main": (0.10, 5.0, 0.05), "lenient": (0.20, 10.0, 0.10)}


# =====================================================================================================================
# 2. Arrival profile and settings
# =====================================================================================================================
@dataclass
class Profile:
    """Stylized two-peak walk-in rate lambda(t) = mean_rate * g(t) on [0, T], with mean(g) = 1 and max(g) = par
    (the peak-to-average ratio). g is a constant base plus two Gaussian bumps (a morning peak and a lower afternoon
    peak); par = 1 gives stationary demand."""
    mean_rate: float
    par: float = 1.6
    peaks: tuple = (90.0, 360.0)
    heights: tuple = (1.0, 0.75)
    width: float = 50.0
    grid_step: float = 0.05
    t: np.ndarray = field(init=False, repr=False)
    cum: np.ndarray = field(init=False, repr=False)

    def __post_init__(self):
        self.t = np.arange(0.0, T + self.grid_step / 2, self.grid_step)
        h = sum(a * np.exp(-0.5 * ((self.t - p) / self.width) ** 2) for a, p in zip(self.heights, self.peaks))
        hbar = np.trapezoid(h, self.t) / T
        if self.par <= 1.0 + 1e-12:
            g = np.ones_like(self.t)
        else:
            base = (h.max() - self.par * hbar) / (self.par - 1.0)
            if base < 0:
                raise ValueError(f"par={self.par} too large for this bump shape (max {h.max() / hbar:.2f})")
            g = (base + h) / (base + hbar)
        self.g = g
        self.cum = np.concatenate([[0.0], np.cumsum(0.5 * (g[1:] + g[:-1]) * np.diff(self.t))]) * self.mean_rate

    def rate(self, t):
        return self.mean_rate * np.interp(t, self.t, self.g)

    def Lambda(self, t):
        """Expected walk-ins in [0, t]."""
        return np.interp(t, self.t, self.cum)

    def arrival_times(self, unit_epochs):
        """Map the epochs of a unit-rate Poisson process to arrival times on [0, T) (time change)."""
        s = unit_epochs[unit_epochs < self.cum[-1]]
        return np.interp(s, self.cum, self.t)

    def block_rates(self, bounds, c=T, kind="avg"):
        """Admitted arrival rate of each block (no arrivals after c). kind='avg': block average; 'max': maximum."""
        out = []
        for a, b in zip(bounds[:-1], bounds[1:]):
            hi = min(b, c)
            if hi <= a:
                out.append(0.0)
            elif kind == "avg":
                out.append((self.Lambda(hi) - self.Lambda(a)) / (b - a))
            else:
                tt = self.t[(self.t >= a) & (self.t <= hi)]
                out.append(float(self.rate(tt).max()))
        return np.array(out)


@dataclass
class Setting:
    """One factor setting: number of blocks, rate profile, service variability, staffing levels, admission time."""
    n_blocks: int = 2
    mean_rate: float = MEAN_RATE
    par: float = 1.6
    cv: tuple = CVS["moderate"]
    levels: int = LEVELS
    cutoffs: tuple = (CUTOFF,)
    rate_mult: float = 1.0
    service_mult: tuple = (1, 1, 1, 1, 1)
    name: str = "custom"

    @property
    def bounds(self):
        return np.linspace(0.0, T, self.n_blocks + 1)

    def profile(self):
        return Profile(self.mean_rate * self.rate_mult, self.par)


def setting(blocks, par, cv, levels=LEVELS):
    return Setting(n_blocks=blocks, mean_rate=MEAN_RATE, par=par, cv=CVS[cv], levels=levels,
                   name=f"B{blocks}-PAR{par}-{cv}")


def grid(blocks_list=BLOCKS, levels=LEVELS):
    return [setting(b, p, c, levels) for b in blocks_list for p in PARS for c in CVS]


def cv_name(s):
    return next((k for k, v in CVS.items() if tuple(float(x) for x in s.cv) == tuple(float(x) for x in v)), "custom")


def tag(s):
    return {"setting": s.name, "blocks": s.n_blocks, "PAR": s.par, "cv": cv_name(s)}


# =====================================================================================================================
# 3. Common random numbers
# =====================================================================================================================
def draw_days(n_days, seed_offset=0):
    """Unit-rate Poisson epochs and standard normal service draws; day i uses seed seed_offset + i + 1."""
    E, Z = np.empty((n_days, MAX_PATIENTS)), np.empty((n_days, MAX_PATIENTS, 5))
    for i in range(n_days):
        rng = np.random.default_rng(seed_offset + i + 1)
        E[i] = np.cumsum(rng.exponential(1.0, MAX_PATIENTS))
        Z[i] = rng.standard_normal((MAX_PATIENTS, 5))
    return E, Z


def scenario_days(E, Z, setting):
    """Arrival times (padded with inf), lognormal service times and admitted counts per cutoff for one setting.
    Service time of patient j at stage k: mean_k * exp(sigma_k Z_jk - sigma_k^2 / 2), sigma_k^2 = ln(1 + cv_k^2)."""
    prof = setting.profile()
    n_days = E.shape[0]
    arr = np.full((n_days, MAX_PATIENTS), INF)
    n_day = np.zeros(n_days, dtype=np.int64)
    for i in range(n_days):
        a = prof.arrival_times(E[i])
        if len(a) >= MAX_PATIENTS:
            raise ValueError("MAX_PATIENTS too small")
        arr[i, :len(a)] = a
        n_day[i] = len(a)
    sig = np.sqrt(np.log1p(np.asarray(setting.cv) ** 2))
    svc = STAGE_MEANS * np.asarray(setting.service_mult, float) * np.exp(sig * Z - 0.5 * sig ** 2)
    n_adm = np.stack([admitted_counts(arr[i], setting.cutoffs) for i in range(n_days)])
    return arr, svc, n_day, n_adm, prof


def admitted_counts(arr_sorted, cutoffs):
    """Patients admitted under each last admission time c: those arriving in [0, c)."""
    return np.searchsorted(arr_sorted, np.asarray(cutoffs, dtype=float), side="left").astype(np.int64)


# =====================================================================================================================
# 4. Reference engine (plain Python)
# =====================================================================================================================
def simulate_day_reference(arr, svc, staff, bounds, trace=None):
    """Plain-Python discrete-event simulation of one day, kept deliberately simple for verification.
    Tie rule at equal times: block boundary, then service completion (earliest started first), then arrival.
    Returns (waits per patient, last completion time per role, extra handover minutes per role). With a list as
    trace, appends one record (patient, stage, role, joined line, start, end) per service."""
    n = len(arr)
    staff = np.asarray(staff)
    B = staff.shape[1]
    cap = [int(staff[r, 0]) for r in range(4)]
    busy = [0] * 4
    queues = [[] for _ in range(4)]
    stage, ready, wait = [0] * n, [0.0] * n, [0.0] * n
    last, handover, lastcut = [0.0] * 4, [0.0] * 4, [0.0] * 4
    events, seq = [], itertools.count()
    for b in range(1, B):
        heapq.heappush(events, (float(bounds[b]), 0, next(seq), "boundary", b, -1))
    for j in range(n):
        heapq.heappush(events, (float(arr[j]), 2, next(seq), "arrival", j, -1))
    joined = [0.0] * n

    def start(j, r, t):
        busy[r] += 1
        end = t + float(svc[j][stage[j]])
        if trace is not None:
            trace.append((j, stage[j], r, joined[j], t, end))
        heapq.heappush(events, (end, 1, next(seq), "done", j, r))

    def join(j, t):
        r = ROUTE[stage[j]]
        joined[j] = t
        if busy[r] < cap[r]:
            start(j, r, t)
        else:
            ready[j] = t
            queues[r].append(j)

    while events:
        t, _, _, kind, j, r = heapq.heappop(events)
        if kind == "boundary":
            b = j
            for r in range(4):
                new = int(staff[r, b])
                if new < cap[r]:
                    lastcut[r] = t
                cap[r] = new
                while busy[r] < cap[r] and queues[r]:
                    k = queues[r].pop(0)
                    wait[k] += t - ready[k]
                    start(k, r, t)
        elif kind == "done":
            busy[r] -= 1
            last[r] = t
            if busy[r] >= cap[r]:                      # this server's block has ended: it leaves
                handover[r] += t - lastcut[r]
            elif queues[r]:
                k = queues[r].pop(0)
                wait[k] += t - ready[k]
                start(k, r, t)
            stage[j] += 1
            if stage[j] < 5:
                join(j, t)
        else:
            join(j, t)
    return np.array(wait), np.array(last), np.array(handover)


# =====================================================================================================================
# 5. Compiled engine (numba); same rules and tie-breaking as the reference engine
# =====================================================================================================================
@njit(cache=True)
def _start(j, r, t, svc, stage, busy, s_end, s_pat, s_role, s_seq, ns, seq):
    busy[r] += 1
    s_end[ns] = t + svc[j, stage[j]]
    s_pat[ns] = j
    s_role[ns] = r
    s_seq[ns] = seq
    return ns + 1, seq + 1


@njit(cache=True)
def _simulate_day(arr, svc, n, staff, bounds, wait, last, handover):
    """Fills wait[:n], last[4], handover[4]. staff: int64 (4, B); bounds: float64 (B + 1)."""
    B = staff.shape[1]
    cap = np.empty(4, np.int64)
    for r in range(4):
        cap[r] = staff[r, 0]
        last[r] = 0.0
        handover[r] = 0.0
    busy = np.zeros(4, np.int64)
    lastcut = np.zeros(4)
    qbuf = np.empty((4, 2 * n + 1), np.int64)
    qh = np.zeros(4, np.int64)
    qt = np.zeros(4, np.int64)
    stage = np.zeros(n, np.int64)
    ready = np.zeros(n)
    for j in range(n):
        wait[j] = 0.0
    MAXS = 4 * 64
    s_end = np.empty(MAXS)
    s_pat = np.empty(MAXS, np.int64)
    s_role = np.empty(MAXS, np.int64)
    s_seq = np.empty(MAXS, np.int64)
    ns = 0
    seq = 0
    ia = 0
    ib = 1
    while True:
        kmin = -1
        tc = np.inf
        for k in range(ns):
            if s_end[k] < tc or (s_end[k] == tc and s_seq[k] < s_seq[kmin]):
                tc = s_end[k]
                kmin = k
        if kmin < 0 and ia >= n:
            break
        ta = arr[ia] if ia < n else np.inf
        tb = bounds[ib] if ib < B else np.inf
        if ib < B and tb <= tc and tb <= ta:
            t = tb
            for r in range(4):
                new = staff[r, ib]
                if new < cap[r]:
                    lastcut[r] = t
                cap[r] = new
                while busy[r] < cap[r] and qh[r] < qt[r]:
                    k = qbuf[r, qh[r]]
                    qh[r] += 1
                    wait[k] += t - ready[k]
                    ns, seq = _start(k, r, t, svc, stage, busy, s_end, s_pat, s_role, s_seq, ns, seq)
            ib += 1
        elif tc <= ta:
            t = tc
            j = s_pat[kmin]
            r = s_role[kmin]
            ns -= 1
            s_end[kmin] = s_end[ns]
            s_pat[kmin] = s_pat[ns]
            s_role[kmin] = s_role[ns]
            s_seq[kmin] = s_seq[ns]
            busy[r] -= 1
            last[r] = t
            if busy[r] >= cap[r]:
                handover[r] += t - lastcut[r]
            elif qh[r] < qt[r]:
                k = qbuf[r, qh[r]]
                qh[r] += 1
                wait[k] += t - ready[k]
                ns, seq = _start(k, r, t, svc, stage, busy, s_end, s_pat, s_role, s_seq, ns, seq)
            stage[j] += 1
            if stage[j] < 5:
                r2 = ROUTE[stage[j]]
                if busy[r2] < cap[r2]:
                    ns, seq = _start(j, r2, t, svc, stage, busy, s_end, s_pat, s_role, s_seq, ns, seq)
                else:
                    ready[j] = t
                    qbuf[r2, qt[r2]] = j
                    qt[r2] += 1
        else:
            t = ta
            j = ia
            ia += 1
            r2 = ROUTE[0]
            if busy[r2] < cap[r2]:
                ns, seq = _start(j, r2, t, svc, stage, busy, s_end, s_pat, s_role, s_seq, ns, seq)
            else:
                ready[j] = t
                qbuf[r2, qt[r2]] = j
                qt[r2] += 1
    return 0


DAY_COLS = ("total_wait", "admitted", "within15", "within20", "within30", "within45", "within60",
            "E0", "E1", "E2", "E3", "H0", "H1", "H2", "H3")


@njit(cache=True)
def _plan_days(arr, svc, n_adm_c, staff, bounds, out):
    """Simulate every day for one plan; out[i] receives the DAY_COLS statistics of day i."""
    n_days = arr.shape[0]
    wait = np.empty(arr.shape[1])
    last = np.empty(4)
    ho = np.empty(4)
    for i in range(n_days):
        n = n_adm_c[i]
        _simulate_day(arr[i], svc[i], n, staff, bounds, wait, last, ho)
        tw = 0.0
        w = np.zeros(5)
        for j in range(n):
            tw += wait[j]
            for q in range(5):
                if wait[j] <= THRESHOLDS[q]:
                    w[q] += 1.0
        out[i, 0] = tw
        out[i, 1] = n
        for q in range(5):
            out[i, 2 + q] = w[q]
        for r in range(4):
            out[i, 7 + r] = last[r]
            out[i, 11 + r] = ho[r]


METRIC_COLS = ("M", "W", "SL", "OT_pay", "O", "wages", "handover_pay", "admitted", "SL_min_day", "compliance",
               "SL15", "SL20", "SL45", "SL60")


@njit(cache=True)
def _metrics_from_days(D, staff, bounds, wages, kappa, cw, horizon, tau, out):
    n_days = D.shape[0]
    B = staff.shape[1]
    wage_bill = 0.0
    for r in range(4):
        for b in range(B):
            wage_bill += wages[r] / horizon * (bounds[b + 1] - bounds[b]) * staff[r, b]
    ot = 0.0
    hp = 0.0
    O = 0.0
    tw = 0.0
    adm = 0.0
    sl = np.zeros(5)
    slmin = 1.0
    comp = 0.0
    for i in range(n_days):
        tw += D[i, 0]
        adm += D[i, 1]
        omax = 0.0
        for r in range(4):
            o = max(0.0, D[i, 7 + r] - horizon)
            ot += kappa * wages[r] / horizon * staff[r, B - 1] * o
            hp += kappa * wages[r] / horizon * D[i, 11 + r]
            omax = max(omax, o)
        O += omax
        for q in range(5):
            s = 1.0 if D[i, 1] == 0 else D[i, 2 + q] / D[i, 1]     # convention: share = 1 on a day without admissions
            sl[q] += s
            if q == 2:
                slmin = min(slmin, s)
                if s >= tau:
                    comp += 1.0
    nd = float(n_days)
    out[0] = wage_bill + (ot + hp) / nd
    out[1] = cw * tw / nd
    out[2] = sl[2] / nd
    out[3] = (ot + hp) / nd
    out[4] = O / nd
    out[5] = wage_bill
    out[6] = hp / nd
    out[7] = adm / nd
    out[8] = slmin
    out[9] = comp / nd
    out[10] = sl[0] / nd
    out[11] = sl[1] / nd
    out[12] = sl[3] / nd
    out[13] = sl[4] / nd


@njit(parallel=True, cache=True)
def _evaluate_batch(arr, svc, n_adm, staffs, cut_idx, bounds, wages, kappa, cw, horizon, tau, out):
    P = staffs.shape[0]
    n_days = arr.shape[0]
    for p in prange(P):
        D = np.empty((n_days, 15))
        _plan_days(arr, svc, n_adm[:, cut_idx[p]], staffs[p], bounds, D)
        _metrics_from_days(D, staffs[p], bounds, wages, kappa, cw, horizon, tau, out[p])


# =====================================================================================================================
# 6. Evaluator
# =====================================================================================================================
class Evaluator:
    """Evaluates plans of one setting on a fixed sample of days (common random numbers) and caches the results.
    A plan is a tuple of 4B staffing levels (role-major: Rec b1..bB, Nur b1..bB, Doc .., Pha ..) plus the cutoff."""

    def __init__(self, setting, n_days=200, seed_offset=0, days=None):
        self.s = setting
        E, Z = days if days is not None else draw_days(n_days, seed_offset)
        self.arr, self.svc, self.n_day, self.n_adm, self.prof = scenario_days(E, Z, setting)
        self.bounds = setting.bounds
        self.B = setting.n_blocks
        self.cache = {}

    def staff_matrix(self, plan):
        return np.asarray(plan[:4 * self.B], dtype=np.int64).reshape(4, self.B)

    def evaluate_many(self, plans):
        """Metric matrix (len(plans), len(METRIC_COLS)); simulates only uncached plans."""
        plans = [tuple(int(v) if i < 4 * self.B else float(v) for i, v in enumerate(p)) for p in plans]
        todo = [p for p in dict.fromkeys(plans) if p not in self.cache]
        if todo:
            staffs = np.stack([self.staff_matrix(p) for p in todo])
            cidx = np.array([list(self.s.cutoffs).index(p[-1]) for p in todo], dtype=np.int64)
            out = np.empty((len(todo), len(METRIC_COLS)))
            _evaluate_batch(self.arr, self.svc, self.n_adm, staffs, cidx, self.bounds, WAGES, KAPPA, C_W, T, TAU, out)
            for p, row in zip(todo, out):
                self.cache[p] = row
        return np.stack([self.cache[p] for p in plans])

    def evaluate_rows(self, rows):
        """Metric matrix for staffing rows (index in product order) of the first cutoff, without the cache."""
        D = staffing_digits(self.s, rows) + 1
        staffs = D.reshape(len(D), 4, self.B).astype(np.int64).copy()
        out = np.empty((len(D), len(METRIC_COLS)))
        _evaluate_batch(self.arr, self.svc, self.n_adm, staffs, np.zeros(len(D), np.int64), self.bounds, WAGES,
                        KAPPA, C_W, T, TAU, out)
        return out

    def day_stats(self, plan):
        """Per-day statistics (n_days, DAY_COLS) of one plan (not cached)."""
        staff = self.staff_matrix(plan)
        c = list(self.s.cutoffs).index(float(plan[-1]))
        D = np.empty((self.arr.shape[0], 15))
        _plan_days(self.arr, self.svc, self.n_adm[:, c].copy(), staff, self.bounds, D)
        return D


# =====================================================================================================================
# 7. Plan geometry and storage of enumerations
# =====================================================================================================================
def plan_columns(setting):
    return [f"{r}{b + 1}" for r in ROLE_NAMES for b in range(setting.n_blocks)] + ["Cutoff"]


def staffing_cols(setting):
    return plan_columns(setting)[:-1]


def geometry(setting):
    nv = 4 * setting.n_blocks
    L = setting.levels
    return nv, L, L ** nv, L ** np.arange(nv - 1, -1, -1)


def staffing_digits(setting, rows):
    """Staffing digits (level - 1) of rows in product order, shape (n, 4B), first column most significant."""
    nv, L, n_stf, pw = geometry(setting)
    return ((np.asarray(rows, np.int64)[:, None] // pw[None, :]) % L).astype(np.int64)


def plan_of_row(row, setting):
    nv, L, n_stf, pw = geometry(setting)
    ci, r = divmod(int(row), n_stf)
    digits = [(r // int(p)) % L + 1 for p in pw]
    return tuple(int(d) for d in digits) + (float(setting.cutoffs[ci]),)


def row_of_plan(plan, setting):
    nv, L, n_stf, pw = geometry(setting)
    ci = list(setting.cutoffs).index(float(plan[-1]))
    return int(ci * n_stf + (np.asarray(plan[:-1], dtype=np.int64) - 1) @ pw)


def keys_of(df, setting):
    cols = plan_columns(setting)
    return [tuple(int(v) for v in r[:-1]) + (float(r[-1]),) for r in df[cols].values.tolist()]


def staffing_text(plan, setting):
    """Readable staffing: receptionists, nurses, physicians, pharmacists per block, e.g. (3,3,3,1 | 2,2,3,2)."""
    x = np.asarray(plan[:4 * setting.n_blocks]).reshape(4, setting.n_blocks)
    return "(" + " | ".join(",".join(str(int(x[r, b])) for r in range(4)) for b in range(setting.n_blocks)) + ")"


STORED = ("M", "W", "SL", "OT_pay")                 # metrics kept on disk for every enumerated plan


def enum_name(s, n_days, seed_offset):
    return f"{s.name}_n{n_days}_s{seed_offset}"


def enum_file(data_dir, s, n_days, seed_offset):
    return os.path.join(data_dir, "enum", enum_name(s, n_days, seed_offset) + ".npz")


def enumeration_jobs(blocks_list=BLOCKS, levels=LEVELS, reliability=True):
    """Every enumeration the paper needs, in the order they are run: main samples of two blocks, main samples of
    three blocks, then the reliability samples of two blocks (four independent 200-day samples and 1,000 days)."""
    jobs = [(s, *MAIN) for b in blocks_list for s in grid((b,), levels)]
    if reliability and 2 in blocks_list:
        jobs += [(s, n, off) for s in grid((2,), levels) for n, off in INDEPENDENT + [REFERENCE]]
    return jobs


def run_enumeration_queue(data_dir, jobs, max_minutes=50.0, target_day_sims=4_000_000, verbose=True):
    """Work through the enumeration jobs in chunks, saving every chunk to disk at once, until all jobs are done or
    max_minutes have passed (a chunk already started is finished). A finished job is merged into one compressed file
    and its chunks are deleted. Rerun to continue: finished chunks and jobs are never recomputed."""
    os.makedirs(os.path.join(data_dir, "enum"), exist_ok=True)
    t0 = time.time()
    done_jobs = 0
    for s, n_days, off in jobs:
        final = enum_file(data_dir, s, n_days, off)
        if os.path.exists(final):
            done_jobs += 1
            continue
        nv, L, n_stf, pw = geometry(s)
        chunk = int(max(1, min(n_stf, math.ceil(target_day_sims / n_days))))
        cdir = os.path.join(data_dir, "enum", "chunks_" + enum_name(s, n_days, off))
        os.makedirs(cdir, exist_ok=True)
        ev = None
        starts = list(range(0, n_stf, chunk))
        for a in starts:
            path = os.path.join(cdir, f"rows_{a:09d}.npy")
            if os.path.exists(path):
                continue
            if (time.time() - t0) / 60.0 > max_minutes:
                if verbose:
                    print(f"Time budget reached after {(time.time() - t0) / 60:.1f} minutes; run this cell again to "
                          f"continue. Jobs finished: {done_jobs} of {len(jobs)}.", flush=True)
                return False
            if ev is None:
                ev = Evaluator(s, n_days, off)
            b = min(a + chunk, n_stf)
            t1 = time.time()
            out = ev.evaluate_rows(np.arange(a, b))
            idx = [METRIC_COLS.index(k) for k in STORED]
            tmp = path + ".tmp.npy"
            np.save(tmp, out[:, idx])
            os.replace(tmp, path)
            if verbose:
                print(f"  {enum_name(s, n_days, off)}: rows {a:,} to {b - 1:,} of {n_stf:,} in {time.time() - t1:.0f}s",
                      flush=True)
        parts = [np.load(os.path.join(cdir, f"rows_{a:09d}.npy")) for a in starts]
        X = np.vstack(parts)
        assert len(X) == n_stf
        tmp = final + ".tmp.npz"
        np.savez_compressed(tmp, metrics=X, columns=np.array(STORED), n_days=n_days, seed_offset=off,
                            setting=s.name)
        os.replace(tmp, final)
        for f in glob.glob(os.path.join(cdir, "*.npy")):
            os.remove(f)
        os.rmdir(cdir)
        done_jobs += 1
        if verbose:
            print(f"Finished {enum_name(s, n_days, off)} ({n_stf:,} staffings, {n_days} days). "
                  f"Jobs finished: {done_jobs} of {len(jobs)}.", flush=True)
    if verbose:
        print(f"All {len(jobs)} enumeration jobs are finished.", flush=True)
    return True


def enumeration_status(data_dir, jobs):
    rows = []
    for s, n, off in jobs:
        nv, L, n_stf, pw = geometry(s)
        final = enum_file(data_dir, s, n, off)
        cdir = os.path.join(data_dir, "enum", "chunks_" + enum_name(s, n, off))
        chunks = len(glob.glob(os.path.join(cdir, "rows_*.npy"))) if os.path.isdir(cdir) else 0
        rows.append({"job": enum_name(s, n, off), "staffings": n_stf, "days": n,
                     "status": "done" if os.path.exists(final) else (f"{chunks} chunks saved" if chunks else "pending")})
    return pd.DataFrame(rows)


def load_table(data_dir, s, n_days=200, seed_offset=0):
    """Enumeration table of one setting and sample: staffing columns, Cutoff, stored metrics and feasibility, one row
    per staffing matrix in product order (row index = staffing index)."""
    z = np.load(enum_file(data_dir, s, n_days, seed_offset))
    X = z["metrics"]
    cols = [str(c) for c in z["columns"]]
    nv, L, n_stf, pw = geometry(s)
    D = (staffing_digits(s, np.arange(n_stf)) + 1).astype(np.int8)
    df = pd.DataFrame(D, columns=staffing_cols(s))
    df["Cutoff"] = float(s.cutoffs[0])
    for k, col in enumerate(cols):
        df[col] = X[:, k]
    df["feasible"] = df.SL >= TAU
    return df


# =====================================================================================================================
# 8. Fronts and structure of the feasible sets
# =====================================================================================================================
def pareto2(M, W):
    """Positions of the non-dominated points of (M, W), both minimized."""
    M, W = np.asarray(M, float), np.asarray(W, float)
    if len(M) == 0:
        return np.array([], np.int64)
    order = np.lexsort((W, M))
    w = W[order]
    prev = np.concatenate([[np.inf], np.minimum.accumulate(w)[:-1]])
    return np.sort(order[w < prev])


def exact_front(df):
    """Exact front of the SAA problem: non-dominated (M, W) among the feasible staffings (rows of df)."""
    idx = np.nonzero(df.feasible.values)[0]
    nd = pareto2(df.M.values[idx], df.W.values[idx])
    return df.iloc[idx[nd]]


def cheapest_row(df):
    """Row index of the cheapest feasible staffing (ties broken by W), or None."""
    f = np.nonzero(df.feasible.values)[0]
    if len(f) == 0:
        return None
    return int(f[np.lexsort((df.W.values[f], df.M.values[f]))[0]])


def minimal_staffings(df, setting):
    """Feasible staffings from which no single staff member can be removed without losing feasibility (the minimal
    elements of an upward-closed feasible set). Returns their row indices."""
    nv, L, n_stf, pw = geometry(setting)
    f = df.feasible.values[:n_stf]
    rows = np.nonzero(f)[0]
    keep = np.ones(len(rows), bool)
    for k in range(nv):
        has = (rows // pw[k]) % L > 0
        below = np.zeros(len(rows), bool)
        below[has] = f[rows[has] - pw[k]]
        keep &= ~below
    return rows[keep]


def product_factor(df, setting):
    """Size of the product of the projections of the feasible set on the blocks divided by the size of the feasible
    set: 1 when feasibility is a condition on each block separately, larger when blocks compensate one another."""
    f = np.nonzero(df.feasible.values)[0]
    if len(f) == 0:
        return np.nan
    D = staffing_digits(setting, f)
    B = setting.n_blocks
    prod = 1
    for b in range(B):
        proj = D[:, [r * B + b for r in range(4)]]
        prod *= len(np.unique(proj, axis=0))
    return prod / len(f)


def single_step_pairs(setting):
    """Row pairs (s, s + one staff member in one role and block), as two arrays."""
    nv, L, n_stf, pw = geometry(setting)
    rows = np.arange(n_stf, dtype=np.int64)
    lo_list, hi_list = [], []
    for k in range(nv):
        digit = (rows // pw[k]) % L
        base = rows[digit < L - 1]
        lo_list.append(base)
        hi_list.append(base + pw[k])
    return np.concatenate(lo_list), np.concatenate(hi_list)


def structure(df, setting):
    """Feasible plans, minimal staffings, product factor, front and cheapest plan of the exact SAA solution."""
    nv, L, n_stf, pw = geometry(setting)
    mins = minimal_staffings(df, setting)
    front = exact_front(df)
    ch = cheapest_row(df)
    row = {**tag(setting), "staffings": n_stf, "feasible": int(df.feasible.sum()),
           "feasible share": float(df.feasible.mean()), "minimal": len(mins), "box": len(mins) == 1,
           "product factor": product_factor(df, setting), "front": len(front),
           "cheapest M": float(df.M.iloc[ch]) if ch is not None else np.nan,
           "cheapest plan": staffing_text(plan_of_row(ch, setting), setting) if ch is not None else "",
           "lowest W on front": float(front.W.min()) if len(front) else np.nan,
           "M of lowest W": float(front.sort_values("W").M.iloc[0]) if len(front) else np.nan}
    return row, mins, front


def day_service_levels(evaluator, plan):
    D = evaluator.day_stats(plan)
    adm = D[:, 1]
    return np.where(adm > 0, D[:, 2 + I_OMEGA] / np.where(adm > 0, adm, 1), 1.0)


def monotonicity(df, setting, evaluator, max_tests=1000):
    """Single-step monotonicity of the sample-mean service level: pairs whose SL decreases when a staff member is
    added, the largest decrease, one-sided paired t-tests on the common days for the largest decreases (Bonferroni
    over all single-step pairs), and crossings, pairs in which adding a staff member makes a feasible plan infeasible
    (each tested, Bonferroni over the crossings)."""
    lo, hi = single_step_pairs(setting)
    SL, feas = df.SL.values, df.feasible.values
    dec = SL[hi] < SL[lo] - 1e-12
    out = {**tag(setting), "pairs": int(len(lo)), "SL decreases": int(dec.sum()),
           "largest SL decrease": float((SL[lo] - SL[hi])[dec].max()) if dec.any() else 0.0}
    nv, L, n_stf, pw = geometry(setting)
    cols = staffing_cols(setting)
    lo_d, hi_d = lo[dec], hi[dec]
    order = np.argsort(-(SL[lo_d] - SL[hi_d]))[:max_tests]
    alpha_b = 0.05 / len(lo)
    sig_roles, sig_sl = [], []
    for i in order:
        d = day_service_levels(evaluator, plan_of_row(hi_d[i], setting)) - \
            day_service_levels(evaluator, plan_of_row(lo_d[i], setting))
        if d.std(ddof=1) == 0:
            continue
        p = stats.t.cdf(d.mean() / (d.std(ddof=1) / math.sqrt(len(d))), len(d) - 1)
        if p < alpha_b:
            k = int(round(math.log(hi_d[i] - lo_d[i]) / math.log(L)))
            sig_roles.append(cols[nv - 1 - k])
            sig_sl.append(SL[lo_d[i]])
    out.update({"tested": int(len(order)), "significant Bonferroni": len(sig_roles),
                "significant where": " ".join(f"{k}:{v}" for k, v in pd.Series(sig_roles, dtype=object)
                                              .value_counts().items()),
                "highest SL among significant": float(max(sig_sl)) if sig_sl else np.nan})
    cross = feas[lo] & ~feas[hi]
    clo, chi = lo[cross], hi[cross]
    ps, gaps = [], []
    for a, b in zip(clo, chi):
        d = day_service_levels(evaluator, plan_of_row(b, setting)) - day_service_levels(evaluator, plan_of_row(a, setting))
        t = d.mean() / (d.std(ddof=1) / math.sqrt(len(d))) if d.std(ddof=1) > 0 else 0.0
        ps.append(stats.t.cdf(t, len(d) - 1))
        gaps.append(SL[a] - SL[b])
    n = len(ps)
    out.update({"crossings": n, "largest crossing drop": max(gaps) if n else np.nan,
                "crossings significant Bonferroni": int(sum(p < 0.05 / n for p in ps)) if n else 0})
    return out


def near_equivalent(df, tolerances=(0.01, 0.02)):
    """Feasible staffings whose labor cost is within each tolerance of the cheapest feasible cost."""
    f = df.feasible.values
    if not f.any():
        return {}
    M = df.M.values
    best = M[f].min()
    return {f"within {int(round(100 * t))}%": int((f & (M <= best * (1 + t))).sum()) for t in tolerances}


# =====================================================================================================================
# 9. Stationary baselines (SIPP) and the modified offered load (MOL)
# =====================================================================================================================
def erlang_c(c, a):
    """Probability of waiting in an M/M/c queue with offered load a < c."""
    term, total = 1.0, 1.0
    for k in range(1, c):
        term *= a / k
        total += term
    tail = term * a / c * c / (c - a)
    return tail / (total + tail)


def station_model(lam, setting, role, servers, visit_rate=None):
    """Stationary wait at one station (Allen-Cunneen GI/G/c with Poisson input): (P(wait), mean wait, conditional
    mean wait given a wait), or None when the station is unstable."""
    stages = [k for k in range(5) if ROUTE[k] == role]
    mult = np.asarray(setting.service_mult, float)
    cv = np.asarray(setting.cv, float)
    means = [STAGE_MEANS[k] * mult[k] for k in stages]
    second = [(STAGE_MEANS[k] * mult[k]) ** 2 * (1 + cv[k] ** 2) for k in stages]
    es, es2 = float(np.mean(means)), float(np.mean(second))
    cs2 = es2 / es ** 2 - 1.0
    Lam = lam * len(stages) if visit_rate is None else visit_rate
    a = Lam * es
    if Lam == 0:
        return 0.0, 0.0, 0.0
    if a >= servers:
        return None
    pw = erlang_c(servers, a)
    wq = pw / (servers / es - Lam) * (1 + cs2) / 2
    return pw, wq, (wq / pw if pw > 0 else 0.0)


def prob_total_wait_le(stations, omega=OMEGA, h=0.02):
    """P(sum of independent stage waits <= omega); each stage wait is 0 w.p. 1-p and exponential with mean m w.p. p."""
    n = int(round(omega / h))
    edges = np.arange(n + 1) * h
    pmf = np.zeros(n + 1)
    pmf[0] = 1.0
    for p, m in stations:
        if p <= 0:
            continue
        cdf = 1.0 - np.exp(-edges / m)
        cell = np.diff(cdf) * p
        q = np.zeros(n + 1)
        q[0] = 1.0 - p
        q[1:] = cell
        pmf = np.convolve(pmf, q)[: n + 1]
    return float(pmf.sum())


def lagged_visit_rates(setting, c):
    """Lag-SIPP visit rates (role x block): each stage sees the admitted arrival rate shifted by the mean time to reach
    it, averaged over the block."""
    prof = setting.profile()
    bounds = setting.bounds
    mult = np.asarray(setting.service_mult, float)
    lags = np.concatenate([[0.0], np.cumsum(STAGE_MEANS[:4] * mult[:4])])

    def adm(t):
        return float(prof.Lambda(min(max(t, 0.0), c)))

    V = np.zeros((4, setting.n_blocks))
    for b, (lo, hi) in enumerate(zip(bounds[:-1], bounds[1:])):
        for k in range(5):
            V[ROUTE[k], b] += (adm(hi - lags[k]) - adm(lo - lags[k])) / (hi - lo)
    return V


def sipp_block_table(setting, kind="avg"):
    """Stationary evaluation of every block staffing (4 roles, levels 1..L) in every block and for every cutoff.
    kind: 'avg' (block-average rate), 'max' (block peak rate) or 'lag' (Lag-SIPP). Returns dict c -> arrays over
    (block, combo): stable, SL_b, mean total wait, per-role waits. combo = sum_r (level_r - 1) L^(3 - r)."""
    prof = setting.profile()
    L, B = setting.levels, setting.n_blocks
    combos = list(itertools.product(range(1, L + 1), repeat=4))
    weights = np.array([L ** 3, L ** 2, L, 1])
    out = {}
    for c in setting.cutoffs:
        lam_b = prof.block_rates(setting.bounds, c=c, kind="avg" if kind == "lag" else kind)
        V = lagged_visit_rates(setting, c) if kind == "lag" else None
        st = np.zeros((B, len(combos)), dtype=bool)
        sl = np.zeros((B, len(combos)))
        tw = np.full((B, len(combos)), np.inf)
        wr = np.full((B, len(combos), 4), np.inf)
        for b in range(B):
            for i, lv in enumerate(combos):
                m = [station_model(lam_b[b], setting, r, lv[r], None if V is None else V[r, b]) for r in range(4)]
                if any(x is None for x in m):
                    continue
                st[b, i] = True
                sl[b, i] = prob_total_wait_le([(m[ROUTE[k]][0], m[ROUTE[k]][2]) for k in range(5)])
                tw[b, i] = sum(m[ROUTE[k]][1] for k in range(5))
                wr[b, i] = [m[r][1] for r in range(4)]
        out[c] = {"stable": st, "SL": sl, "wait": tw, "role_wait": wr, "lam": lam_b, "weights": weights}
    return out


def sipp_table(setting, kind="avg", plans=None):
    """Plan-by-plan stationary evaluation (used only to test the vectorized screens)."""
    prof = setting.profile()
    bounds = setting.bounds
    B, L = setting.n_blocks, setting.levels
    mult = np.asarray(setting.service_mult, float)
    X = np.asarray(plans, dtype=float)
    S = X[:, :4 * B].astype(int).reshape(-1, 4, B)
    cut = X[:, -1]
    tab = sipp_block_table(setting, kind)
    n = len(X)
    stable = np.zeros(n, dtype=bool)
    sl_daily, sl_block = np.zeros(n), np.zeros(n)
    W_hat, M_hat = np.full(n, np.inf), np.full(n, np.inf)
    blen = np.diff(bounds)
    wages = np.einsum("prb,r,b->p", S, WAGES / T, blen)
    for c in setting.cutoffs:
        idx = np.nonzero(cut == c)[0]
        if len(idx) == 0:
            continue
        t = tab[c]
        combo = np.einsum("prb,r->pb", S[idx] - 1, t["weights"])
        N_b = np.array([prof.Lambda(min(b2, c)) - prof.Lambda(min(b1, c)) for b1, b2 in zip(bounds[:-1], bounds[1:])])
        act = N_b > 0
        st = np.ones(len(idx), dtype=bool)
        slb = np.ones((len(idx), B))
        wsum = np.zeros(len(idx))
        for b in range(B):
            if not act[b]:
                continue
            st &= t["stable"][b, combo[:, b]]
            slb[:, b] = t["SL"][b, combo[:, b]]
            wsum += N_b[b] * t["wait"][b, combo[:, b]]
        lam_c = float(prof.rate(max(c - 1e-6, 0.0)))
        t_last = c - (1.0 / lam_c if lam_c > 0 else 0.0)
        bl = min(max(int(np.searchsorted(bounds, t_last, side="right") - 1), 0), B - 1)
        rw = t["role_wait"][bl, combo[:, bl]]
        tt = np.full(len(idx), t_last)
        release = np.zeros((len(idx), 4))
        for k in range(5):
            r = ROUTE[k]
            tt = tt + rw[:, r] + STAGE_MEANS[k] * mult[k]
            release[:, r] = tt
        ot = KAPPA * np.einsum("pr,pr->p", np.maximum(0.0, release - T), (WAGES / T) * S[idx, :, B - 1])
        stable[idx] = st
        sl_daily[idx] = np.where(st, (slb * N_b).sum(1) / N_b.sum(), 0.0)
        sl_block[idx] = np.where(st, slb[:, act].min(1), 0.0)
        W_hat[idx] = np.where(st, C_W * wsum, np.inf)
        M_hat[idx] = np.where(st, wages[idx] + ot, np.inf)
    out = pd.DataFrame(X[:, :4 * B].astype(int), columns=plan_columns(setting)[:-1])
    out["Cutoff"] = cut
    out["stable"], out["SL_daily_hat"], out["SL_block_hat"] = stable, sl_daily, sl_block
    out["M_hat"], out["W_hat"] = M_hat, W_hat
    return out


def sipp_block_monotone(setting, kinds=("avg", "lag", "max")):
    """Check that adding one staff member never makes a stable block unstable and never lowers its stationary
    service level (premise of the product structure of the per-block rule)."""
    L = setting.levels
    rows = []
    for k in kinds:
        tab = sipp_block_table(setting, k)
        for c, t in tab.items():
            for b in range(setting.n_blocks):
                viol_sl = viol_st = 0
                for i in range(L ** 4):
                    lv = [(i // L ** (3 - r)) % L for r in range(4)]
                    for r in range(4):
                        if lv[r] < L - 1:
                            j = i + L ** (3 - r)
                            if t["stable"][b, i] and not t["stable"][b, j]:
                                viol_st += 1
                            if t["stable"][b, i] and t["SL"][b, j] < t["SL"][b, i] - 1e-12:
                                viol_sl += 1
                rows.append({**tag(setting), "kind": k, "block": b + 1, "stability violations": viol_st,
                             "SL decreases": viol_sl})
    return pd.DataFrame(rows)


GRID = 720                     # minutes covered by the offered-load grid
H_STEP = 0.02                  # step of the wait convolution
N_FFT = 8192                   # FFT length (at least 5 * OMEGA / H_STEP + 1, so the circular convolution does not wrap)


def lognormal_params(mean, cv):
    s2 = math.log1p(cv ** 2)
    return math.log(mean) - s2 / 2, math.sqrt(s2)


def integrated_cdf(y, mean, cv):
    """H(y) = int_0^y P(S <= x) dx = y F(y) - E[S; S <= y] for a lognormal S."""
    mu, sig = lognormal_params(mean, cv)
    y = np.asarray(y, float)
    out = np.zeros_like(y)
    pos = y > 0
    ly = np.log(y[pos])
    out[pos] = y[pos] * ndtr((ly - mu) / sig) - mean * ndtr((ly - mu - sig ** 2) / sig)
    return out


def stage_kernels(mean, cv, horizon=GRID, H=None):
    """Kernels of an infinite-server stage on the 1-minute grid, arrivals spread uniformly within each minute:
    G[i] = int_i^{i+1} P(S > u) du and q[d], the probability that an arrival of minute j departs in minute j + d."""
    Hv = integrated_cdf(np.arange(horizon + 2), mean, cv) if H is None else H(np.arange(horizon + 2))
    G = 1.0 - (Hv[1:] - Hv[:-1])
    q = Hv[1:] - 2 * Hv[:-1] + np.concatenate([[0.0], Hv[:-2]])
    return G, q


def offered_loads_from_input(a, kernels, horizon=GRID):
    """Offered loads m_k(t), t = 0..horizon, of a tandem of infinite-server stages fed by a[j] arrivals in minute j."""
    K = len(kernels)
    loads = np.zeros((K, horizon + 1))
    inputs = np.zeros((K, horizon))
    inp = np.asarray(a, float)[:horizon]
    for k, (G, q) in enumerate(kernels):
        inputs[k] = inp
        loads[k, 1:] = np.convolve(inp, G)[:horizon]
        inp = np.convolve(inp, q)[:horizon]
    return loads, inputs


def admissions_per_minute(setting, c, horizon=GRID):
    prof = setting.profile()
    t = np.arange(horizon + 1, dtype=float)
    return prof.Lambda(np.minimum(t[1:], c)) - prof.Lambda(np.minimum(t[:-1], c))


def offered_loads(setting, c, horizon=GRID):
    """Offered loads of the five stages (rows) at t = 0..horizon."""
    mult = np.asarray(setting.service_mult, float)
    kern = [stage_kernels(STAGE_MEANS[k] * mult[k], setting.cv[k], horizon) for k in range(5)]
    return offered_loads_from_input(admissions_per_minute(setting, c, horizon), kern, horizon)[0]


def station_moments(setting, role):
    stages = [k for k in range(5) if ROUTE[k] == role]
    mult = np.asarray(setting.service_mult, float)
    cv = np.asarray(setting.cv, float)
    means = [STAGE_MEANS[k] * mult[k] for k in stages]
    second = [(STAGE_MEANS[k] * mult[k]) ** 2 * (1 + cv[k] ** 2) for k in stages]
    es, es2 = float(np.mean(means)), float(np.mean(second))
    return es, es2 / es ** 2 - 1.0, len(stages)


def erlang_c_vec(s, a):
    a = np.asarray(a, float)
    term = np.ones_like(a)
    total = np.ones_like(a)
    for k in range(1, s):
        term = term * a / k
        total = total + term
    with np.errstate(divide="ignore", invalid="ignore"):
        tail = term * a / (s - a)
        return tail / (total + tail)


def station_arrays(load, s, es, cs2):
    """Allen-Cunneen quantities at offered load `load` with s servers: stable, P(wait), mean wait, conditional mean."""
    load = np.asarray(load, float)
    stable = load < s
    with np.errstate(divide="ignore", invalid="ignore"):
        pw = np.where(stable, erlang_c_vec(s, np.where(stable, load, 0.0)), 1.0)
        wq = np.where(stable, pw * es / (s - load) * (1 + cs2) / 2, np.inf)
        cond = np.where(stable, np.where(pw > 0, wq / np.where(pw > 0, pw, 1.0), 0.0), np.inf)
    return stable, pw, wq, cond


def _sum_weights(n, N=N_FFT):
    u = np.zeros(N)
    u[: n + 1] = 1.0
    U = np.fft.rfft(u)
    w = np.conj(U) / N
    w[1:-1] *= 2.0
    return w


def stage_spectra(stable, pw, cond, h=H_STEP, omega=OMEGA, N=N_FFT):
    n = int(round(omega / h))
    shape = np.shape(pw)
    edges = np.arange(n + 1) * h
    q = np.zeros(shape + (N,))
    p = np.where(stable, pw, 0.0)
    m = np.where(stable & (cond > 0), cond, 1.0)
    cells = -np.diff(np.exp(-edges[None, :] / m.reshape(-1, 1)), axis=1).reshape(shape + (n,))
    q[..., 0] = np.where(stable, 1.0 - p, 0.0)
    q[..., 1:n + 1] = np.where(stable[..., None], p[..., None] * cells, 0.0)
    return np.fft.rfft(q, axis=-1), n


def path_service_levels(spectra, n, N=N_FFT):
    w = _sum_weights(n, N)
    P = spectra[0]
    for k in range(1, spectra.shape[0]):
        P = (P[:, None, :] * spectra[k][None, :, :]).reshape(-1, spectra.shape[-1])
    return np.clip((P @ w).real, 0.0, 1.0)


class MOL:
    """MOL tables of one setting and last admission time c: offered loads of the infinite-server clinic that opens
    empty, each station evaluated as a stationary queue at the load of the instant a patient reaches it; a patient
    whose path meets an unstable station gets service level 0."""

    def __init__(self, setting, c, horizon=GRID):
        self.s, self.c, self.L = setting, float(c), setting.levels
        L = self.L
        mult = np.asarray(setting.service_mult, float)
        self.means = STAGE_MEANS * mult
        self.lags = np.concatenate([[0.0], np.cumsum(self.means[:4])])
        loads = offered_loads(setting, c, horizon)
        self.grid = np.arange(horizon + 1, dtype=float)
        self.role_load = np.vstack([loads[0] + loads[4], loads[1], loads[2], loads[3]])
        self.mom = [station_moments(setting, r) for r in range(4)]
        a = admissions_per_minute(setting, c, horizon)
        nc = int(math.ceil(c))
        self.adm = a[:nc]
        self.tau = np.arange(nc)[:, None] + 0.5 + self.lags[None, :]
        st = np.zeros((nc, 5, L), bool)
        pw = np.zeros((nc, 5, L))
        wq = np.zeros((nc, 5, L))
        cond = np.zeros((nc, 5, L))
        for k in range(5):
            r = ROUTE[k]
            ld = np.interp(self.tau[:, k], self.grid, self.role_load[r])
            es, cs2, _ = self.mom[r]
            for s in range(1, L + 1):
                st[:, k, s - 1], pw[:, k, s - 1], wq[:, k, s - 1], cond[:, k, s - 1] = station_arrays(ld, s, es, cs2)
        self.stable, self.pw, self.wq, self.cond = st, pw, wq, cond
        self.SL = np.empty((nc, L ** 5))
        for a0 in range(0, nc, 16):
            b0 = min(a0 + 16, nc)
            spec, n = stage_spectra(st[a0:b0], pw[a0:b0], cond[a0:b0])
            for j in range(a0, b0):
                self.SL[j] = path_service_levels(spec[j - a0], n)
        self.OT = self._overtime_table()
        self._by_blocks = {}

    def load_at(self, role, t):
        return np.interp(t, self.grid, self.role_load[role])

    def _overtime_table(self):
        prof = self.s.profile()
        lam_c = float(prof.rate(max(self.c - 1e-6, 0.0)))
        t_last = self.c - (1.0 / lam_c if lam_c > 0 else 0.0)
        L = self.L
        out = np.empty(L ** 4)
        for i, lv in enumerate(itertools.product(range(1, L + 1), repeat=4)):
            tt, release, ok = t_last, np.zeros(4), True
            for k in range(5):
                r = ROUTE[k]
                es, cs2, _ = self.mom[r]
                stb, _, w, _ = station_arrays(self.load_at(r, tt), lv[r], es, cs2)
                if not stb:
                    ok = False
                    break
                tt = tt + float(w) + self.means[k]
                release[r] = tt
            out[i] = (KAPPA * (np.maximum(0.0, release - T) * (WAGES / T) * np.array(lv)).sum() if ok else np.inf)
        return out

    def for_blocks(self, B):
        if B in self._by_blocks:
            return self._by_blocks[B]
        blen = T / B
        beta = np.minimum((self.tau // blen).astype(int), B - 1)
        groups, inv = np.unique(beta, axis=0, return_inverse=True)
        inv = np.asarray(inv).reshape(-1)
        Tg = np.zeros((len(groups), self.L ** 5))
        np.add.at(Tg, inv, self.adm[:, None] * self.SL)
        Wk = np.zeros((5, B, self.L))
        for k in range(5):
            for b in range(B):
                m = beta[:, k] == b
                if m.any():
                    Wk[k, b] = (self.adm[m, None] * self.wq[m, k, :]).sum(0)
        res = (groups, Tg, Wk)
        self._by_blocks[B] = res
        return res

    def patient_levels(self, staff):
        staff = np.asarray(staff)
        B = staff.shape[1]
        beta = np.minimum((self.tau // (T / B)).astype(int), B - 1)
        combo = np.zeros(len(self.adm), np.int64)
        for k in range(5):
            combo = combo * self.L + (staff[ROUTE[k], beta[:, k]] - 1)
        return self.SL[np.arange(len(self.adm)), combo]

    def patient_waits(self, staff):
        staff = np.asarray(staff)
        B = staff.shape[1]
        beta = np.minimum((self.tau // (T / B)).astype(int), B - 1)
        out = np.zeros(len(self.adm))
        for k in range(5):
            out += self.wq[np.arange(len(self.adm)), k, staff[ROUTE[k], beta[:, k]] - 1]
        return out


class Screens:
    """Every analytic baseline of one setting, applied to plans given as staffing digits (n, 4B). evaluate() returns,
    per variant, the feasibility mask and the estimates SL, M and W (inf when the baseline declares instability)."""

    def __init__(self, setting, kinds=("avg", "lag", "max"), mol=True):
        self.s = setting
        self.B, self.L = setting.n_blocks, setting.levels
        prof = setting.profile()
        bounds = setting.bounds
        self.blen = np.diff(bounds)
        self.kinds = kinds
        self.sipp = {k: sipp_block_table(setting, k) for k in kinds}
        self.N = {c: np.array([prof.Lambda(min(b2, c)) - prof.Lambda(min(b1, c))
                               for b1, b2 in zip(bounds[:-1], bounds[1:])]) for c in setting.cutoffs}
        self.ot = {k: {c: self._sipp_overtime(k, c) for c in setting.cutoffs} for k in kinds}
        self.mol = {c: MOL(setting, c) for c in setting.cutoffs} if mol else {}

    def _sipp_overtime(self, kind, c):
        prof = self.s.profile()
        bounds = self.s.bounds
        mult = np.asarray(self.s.service_mult, float)
        lam_c = float(prof.rate(max(c - 1e-6, 0.0)))
        t_last = c - (1.0 / lam_c if lam_c > 0 else 0.0)
        bl = min(max(int(np.searchsorted(bounds, t_last, side="right") - 1), 0), self.B - 1)
        rw = self.sipp[kind][c]["role_wait"][bl]
        lv = np.array(list(itertools.product(range(1, self.L + 1), repeat=4)), float)
        tt = np.full(len(lv), t_last)
        release = np.zeros((len(lv), 4))
        for k in range(5):
            r = ROUTE[k]
            tt = tt + rw[:, r] + STAGE_MEANS[k] * mult[k]
            release[:, r] = tt
        with np.errstate(invalid="ignore"):
            return KAPPA * (np.maximum(0.0, release - T) * (WAGES / T) * lv).sum(1)

    def wages(self, D):
        lv = D.reshape(len(D), 4, self.B) + 1
        return np.einsum("prb,r,b->p", lv, WAGES / T, self.blen)

    def block_combos(self, D):
        B, L = self.B, self.L
        out = np.zeros((len(D), B), np.int64)
        for r in range(4):
            out = out * L + D[:, r * B:(r + 1) * B]
        return out

    def evaluate(self, D, c=CUTOFF, variants=VARIANTS):
        D = np.asarray(D, np.int64)
        c = float(c)
        B = self.B
        res = {}
        wages = self.wages(D)
        cb = self.block_combos(D)
        N = self.N[c]
        act = N > 0
        for k in self.kinds:
            if not any(v.startswith(k + "-") for v in variants):
                continue
            t = self.sipp[k][c]
            st = np.ones(len(D), bool)
            slb = np.ones((len(D), B))
            wsum = np.zeros(len(D))
            for b in range(B):
                if not act[b]:
                    continue
                st &= t["stable"][b, cb[:, b]]
                slb[:, b] = t["SL"][b, cb[:, b]]
                wsum = wsum + N[b] * t["wait"][b, cb[:, b]]
            daily = np.where(st, (slb * N).sum(1) / N.sum(), 0.0)
            block = np.where(st, slb[:, act].min(1), 0.0)
            M = np.where(st, wages + self.ot[k][c][cb[:, B - 1]], np.inf)
            W = np.where(st, C_W * wsum, np.inf)
            if f"{k}-block" in variants:
                res[f"{k}-block"] = {"feas": st & (block >= TAU), "SL": block, "M": M, "W": W}
            if f"{k}-daily" in variants:
                res[f"{k}-daily"] = {"feas": st & (daily >= TAU), "SL": daily, "M": M, "W": W}
        if "mol-daily" in variants and self.mol:
            mol = self.mol[c]
            groups, Tg, Wk = mol.for_blocks(B)
            sl = np.zeros(len(D))
            for g, blocks in enumerate(groups):
                combo = np.zeros(len(D), np.int64)
                for k in range(5):
                    combo = combo * self.L + D[:, ROUTE[k] * B + blocks[k]]
                sl += Tg[g, combo]
            sl /= mol.adm.sum()
            W = np.zeros(len(D))
            for k in range(5):
                for b in range(B):
                    W = W + Wk[k, b, D[:, ROUTE[k] * B + b]]
            M = wages + mol.OT[cb[:, B - 1]]
            res["mol-daily"] = {"feas": sl >= TAU, "SL": sl, "M": M, "W": C_W * W}
        return res


def evaluate_all_baselines(setting, variants=VARIANTS):
    """Every analytic baseline on every staffing of the setting (first cutoff), computed once."""
    nv, L, n_stf, pw = geometry(setting)
    D = staffing_digits(setting, np.arange(n_stf))
    scr = Screens(setting, mol="mol-daily" in variants)
    return scr.evaluate(D, setting.cutoffs[0], variants), scr


def peakedness(setting, c=CUTOFF):
    """Within-block peakedness: the largest ratio, over blocks, of a block's peak admitted rate to its average."""
    prof = setting.profile()
    avg = prof.block_rates(setting.bounds, c=c, kind="avg")
    peak = prof.block_rates(setting.bounds, c=c, kind="max")
    return float(np.max(np.where(avg > 0, peak / np.where(avg > 0, avg, 1.0), 1.0)))


# =====================================================================================================================
# 10. Decision-level comparison and error classes
# =====================================================================================================================
def error_class(front_rejected, has_pick, pick_feasible, pick_premium, own_failing_share,
                thresholds=THRESHOLD_SETS["main"]):
    """Conservative: rejects at least rej of the reference front, or its feasible pick costs at least prem% more than
    the cheapest feasible plan. Optimistic: its pick fails the standard, or at least fail of its own front fails.
    No plan: the baseline accepts no staffing (counted as conservative)."""
    rej, prem, fail = thresholds
    if not has_pick:
        return "no plan"
    cons = (front_rejected >= rej - 1e-12) or (pick_feasible and pick_premium >= prem - 1e-12)
    opt = (not pick_feasible) or (own_failing_share is not None and not pd.isna(own_failing_share)
                                  and own_failing_share >= fail - 1e-12)
    return {(True, True): "both", (True, False): "conservative", (False, True): "optimistic",
            (False, False): "neither"}[(bool(cons), bool(opt))]


def compare_baselines(df, setting, res, variants=VARIANTS):
    """Every analytic baseline against the exact SAA solution in df (any sample): share of the exact front it
    rejects, share of the staffings it accepts that fail in simulation, its own front (non-dominated on its own
    estimates) and the share of it that fails, and its pick (lowest estimated M, ties by estimated W) with the pick's
    simulated service level and cost premium over the cheapest feasible plan."""
    feas, M, W, SL = df.feasible.values, df.M.values, df.W.values, df.SL.values
    front = exact_front(df)
    frows = front.index.values
    ch = cheapest_row(df)
    best = float(M[ch]) if ch is not None else np.nan
    rows, own = [], []
    for v in variants:
        r = res[v]
        adm = r["feas"]
        idx = np.nonzero(adm)[0]
        row = {**tag(setting), "variant": v, "label": VARIANT_LABEL[v], "accepted": int(adm.sum()),
               "accepted failing share": float((~feas[adm]).mean()) if adm.any() else np.nan,
               "front": len(front), "front rejected": float(1 - adm[frows].mean()) if len(frows) else np.nan,
               "cheapest M": best}
        if len(idx):
            nd = idx[pareto2(r["M"][idx], r["W"][idx])]
            ok = feas[nd]
            j = int(idx[np.lexsort((r["W"][idx], r["M"][idx]))[0]])
            row.update({"own front": len(nd), "own front failing": int((~ok).sum()),
                        "own front failing share": float((~ok).mean()),
                        "pick row": j, "pick": staffing_text(plan_of_row(j, setting), setting),
                        "pick M_hat": float(r["M"][j]), "pick SL_hat": float(r["SL"][j]),
                        "pick SL": float(SL[j]), "pick feasible": bool(feas[j]), "pick M": float(M[j]),
                        "pick W": float(W[j]), "pick premium %": 100 * (M[j] - best) / best})
            own += [{**tag(setting), "variant": v, "row": int(i), "M_hat": r["M"][i], "W_hat": r["W"][i],
                     "M": M[i], "W": W[i], "SL": SL[i], "feasible": bool(feas[i])} for i in nd]
        else:
            row.update({"own front": 0, "own front failing": 0, "own front failing share": np.nan, "pick row": -1,
                        "pick": "", "pick M_hat": np.nan, "pick SL_hat": np.nan, "pick SL": np.nan,
                        "pick feasible": False, "pick M": np.nan, "pick W": np.nan, "pick premium %": np.nan})
        for name, th in THRESHOLD_SETS.items():
            row[f"class {name}"] = error_class(row["front rejected"], len(idx) > 0, row["pick feasible"],
                                               row["pick premium %"], row["own front failing share"], th)
        rows.append(row)
    return pd.DataFrame(rows), pd.DataFrame(own)


# =====================================================================================================================
# 11. Two-dimensional indicators
# =====================================================================================================================
def _hv2d(pts, ref):
    pts = sorted(p for p in pts if p[0] < ref[0] and p[1] < ref[1])
    area, best_y = 0.0, ref[1]
    for x, y in pts:
        if y < best_y:
            area += (ref[0] - x) * (best_y - y)
            best_y = y
    return area


def hv2(points, ref):
    """Exact hypervolume (area) dominated by points in the plane, both objectives minimized, bounded by ref."""
    return _hv2d([tuple(p) for p in np.asarray(points, float).reshape(-1, 2)], tuple(ref))


class Reference:
    """Reference front (plan keys and (M, W)); indicators in the plane rescaled by its ideal and nadir points."""

    def __init__(self, keys, F):
        self.keys = list(keys)
        self.keyset = set(self.keys)
        self.F = np.asarray(F, dtype=float).reshape(-1, 2)
        self.ideal = self.F.min(0)
        self.scale = np.maximum(self.F.max(0) - self.ideal, 1e-9)
        self.Fn = (self.F - self.ideal) / self.scale
        self.ref = np.full(2, 1.1)
        self.hv = hv2(self.Fn, self.ref)
        self.size = len(self.keys)

    def normalize(self, F):
        return (np.asarray(F, float).reshape(-1, 2) - self.ideal) / self.scale

    def hv_ratio(self, F):
        if len(F) == 0:
            return 0.0
        return hv2(self.normalize(F), self.ref) / self.hv

    def igd(self, F):
        if len(F) == 0:
            return np.inf
        A = self.normalize(F)
        return float(np.mean([np.min(np.linalg.norm(A - f, axis=1)) for f in self.Fn]))


# =====================================================================================================================
# 12. Descent from an analytic plan
# =====================================================================================================================
class TableOracle:
    """Plays the simulator on an enumerated table: returns (M, W, SL) of plans and records the distinct plans
    requested, in order (the simulation budget a run would spend)."""

    def __init__(self, df, setting):
        self.s = setting
        self.V = df[["M", "W", "SL"]].values
        self.order, self.vals = [], {}

    def values(self, plans):
        out = []
        for p in plans:
            if p not in self.vals:
                self.vals[p] = self.V[row_of_plan(p, self.s)]
                self.order.append(p)
            out.append(self.vals[p])
        return np.array(out).reshape(len(plans), 3)


class SimOracle(TableOracle):
    """Same interface, backed by simulation on an evaluator's days."""

    def __init__(self, evaluator, setting):
        self.s, self.ev = setting, evaluator
        self.order, self.vals = [], {}
        self.idx = [METRIC_COLS.index(k) for k in ("M", "W", "SL")]

    def values(self, plans):
        new = [p for p in dict.fromkeys(plans) if p not in self.vals]
        if new:
            X = self.ev.evaluate_many(new)[:, self.idx]
            for p, x in zip(new, X):
                self.vals[p] = x
                self.order.append(p)
        return np.array([self.vals[p] for p in plans]).reshape(len(plans), 3)


def wage_of_staff(setting):
    """Wage of one staff member in each variable (role-major: role r, block b), over its block."""
    blen = np.diff(setting.bounds)
    return np.array([WAGES[r] / T * blen[b] for r in range(4) for b in range(setting.n_blocks)])


def first_improvement(oracle, setting, start, rng):
    """First-improvement descent from a feasible plan. Moves: remove one staff member, or remove one and add one
    elsewhere when that saves wages; a move is accepted when the new plan is feasible and its simulated cost M is
    lower. Moves are scanned in a random order, redrawn at every step. Returns the local optimum."""
    nv = 4 * setting.n_blocks
    L = setting.levels
    sav = wage_of_staff(setting)
    x = start
    fx = oracle.values([x])[0, 0]
    while True:
        moves = [(k, None) for k in range(nv) if x[k] > 1]
        moves += [(k, j) for k in range(nv) for j in range(nv) if j != k and x[k] > 1 and x[j] < L
                  and sav[k] - sav[j] > 1e-9]
        improved = False
        for i in rng.permutation(len(moves)):
            k, j = moves[i]
            y = list(x)
            y[k] -= 1
            if j is not None:
                y[j] += 1
            y = tuple(y)
            v = oracle.values([y])[0]
            if v[2] >= TAU and v[0] < fx - 1e-9:
                x, fx, improved = y, v[0], True
                break
        if not improved:
            return x


def repair_upward(oracle, setting, start):
    """From an infeasible plan, add the staff member that raises the simulated service level most, until feasible
    (or full staffing). Returns the feasible plan, or None if full staffing is infeasible."""
    nv = 4 * setting.n_blocks
    L = setting.levels
    x = start
    while oracle.values([x])[0, 2] < TAU:
        cand = [x[:k] + (x[k] + 1,) + x[k + 1:] for k in range(nv) if x[k] < L]
        if not cand:
            return None
        V = oracle.values(cand)
        x = cand[int(np.argmax(V[:, 2]))]
    return x


# =====================================================================================================================
# 13. Sampling and reliability
# =====================================================================================================================
def sl_interval(evaluator, plan, level=0.95):
    s = day_service_levels(evaluator, plan)
    n = len(s)
    h = stats.t.ppf(0.5 + level / 2, n - 1) * s.std(ddof=1) / math.sqrt(n)
    return float(s.mean()), float(s.mean() - h), float(s.mean() + h)


def jaccard(a, b):
    a, b = set(a), set(b)
    return len(a & b) / len(a | b) if (a | b) else 1.0


# =====================================================================================================================
# 14. Mechanism and verification
# =====================================================================================================================
def waits_by_arrival(evaluator, plan, bin_minutes=10):
    """Mean total wait of the admitted patients by arrival time (bins of bin_minutes), over the evaluator's days."""
    S = evaluator.staff_matrix(plan)
    ci = list(evaluator.s.cutoffs).index(float(plan[-1]))
    nb = int(math.ceil(T / bin_minutes))
    sums, counts = np.zeros(nb), np.zeros(nb)
    w, last, ho = np.empty(MAX_PATIENTS), np.empty(4), np.empty(4)
    for i in range(evaluator.arr.shape[0]):
        n = evaluator.n_adm[i, ci]
        _simulate_day(evaluator.arr[i], evaluator.svc[i], n, S, evaluator.bounds, w, last, ho)
        b = np.minimum((evaluator.arr[i, :n] // bin_minutes).astype(int), nb - 1)
        np.add.at(sums, b, w[:n])
        np.add.at(counts, b, 1)
    return np.arange(nb) * bin_minutes + bin_minutes / 2, sums / np.maximum(counts, 1)


def erlang_c_check(n_runs=30, horizon=200_000.0, warm=10_000.0, lam=MEAN_RATE,
                   staffings=((2, 2, 2, 1), (3, 2, 3, 2))):
    """Exponential services, one block, constant rate, no cutoff: the clinic is a Kelly network, so the mean wait per
    visit at each station must match the Erlang C value at the station's total visit rate. Stage waits come from the
    reference engine's trace (the compiled engine matches it exactly, see the unit tests)."""
    names = ["Reception (check-in)", "Nurse", "Physician", "Pharmacy", "Reception (checkout)"]
    rows = []
    for staff in staffings:
        R = np.zeros((n_runs, 5))
        for rep in range(n_runs):
            rng = np.random.default_rng(700_000 + rep)
            arr = np.cumsum(rng.exponential(1 / lam, int(horizon * lam * 1.3)))
            arr = arr[arr < horizon]
            svc = rng.exponential(1.0, (len(arr), 5)) * STAGE_MEANS
            trace = []
            simulate_day_reference(arr, svc, np.array(staff).reshape(4, 1), np.array([0.0, horizon]), trace=trace)
            tr = np.array(trace)
            keep = (arr[tr[:, 0].astype(int)] > warm) & (arr[tr[:, 0].astype(int)] < horizon - warm)
            for k in range(5):
                m = keep & (tr[:, 1] == k)
                R[rep, k] = (tr[m, 4] - tr[m, 3]).mean()
        tcrit = stats.t.ppf(0.975, n_runs - 1)
        for k in range(5):
            r_ = ROUTE[k]
            visits = 2 if r_ == 0 else 1
            mean_s, c = STAGE_MEANS[k], staff[r_]
            a = visits * lam * mean_s
            tail = a ** c / math.factorial(c) / (1 - a / c)
            pw = tail / (sum(a ** j / math.factorial(j) for j in range(c)) + tail)
            theory = pw / (c / mean_s - visits * lam)
            m, h = R[:, k].mean(), tcrit * R[:, k].std(ddof=1) / math.sqrt(n_runs)
            rows.append({"staffing": str(tuple(staff)), "station": names[k], "servers": c, "utilization": a / c,
                         "simulated": m, "CI low": m - h, "CI high": m + h, "Erlang C": theory,
                         "relative error %": 100 * (m - theory) / theory, "inside CI": m - h <= theory <= m + h})
    return pd.DataFrame(rows)


# =====================================================================================================================
# 15. Output helpers: paths, LaTeX tables, figures
# =====================================================================================================================
def make_paths(base):
    p = {"base": base, "core": os.path.join(base, "core"), "data": os.path.join(base, "data"),
         "tables": os.path.join(base, "tables"), "figures": os.path.join(base, "figures"),
         "results": os.path.join(base, "results")}
    for v in p.values():
        os.makedirs(v, exist_ok=True)
    return p


def latex_table(colspec, header, body, caption, label, width=r"\textwidth"):
    """Table in adjustbox with a pipe-delimited tabular."""
    lines = [r"\begin{table}[htbp]", r"\centering",
             rf"\begin{{adjustbox}}{{max width={width}, max totalheight=0.88\textheight}}",
             rf"\begin{{tabular}}{{{colspec}}}", r"\hline", header, r"\hline"] + list(body) + [
        r"\hline", r"\end{tabular}", r"\end{adjustbox}", rf"\caption{{{caption}}}", rf"\label{{{label}}}",
        r"\end{table}"]
    return "\n".join(lines) + "\n"


def write_text(path, text):
    with open(path, "w") as f:
        f.write(text)
    return path


SURFACE = "#fcfcfb"
INK, INK2, MUTED, GRIDC = "#0b0b0b", "#52514e", "#8a8984", "#e4e3df"
CAT = {"blue": "#2a78d6", "orange": "#eb6834", "aqua": "#1baf7a", "gray": "#52514e"}
ORDINAL = ["#86b6ef", "#3987e5", "#256abf", "#104281"]


def fig_style():
    import matplotlib.pyplot as plt
    plt.rcParams.update({
        "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white",
        "axes.edgecolor": MUTED, "axes.linewidth": 0.6, "axes.labelcolor": INK2, "axes.titlesize": 10,
        "axes.titlecolor": INK, "axes.labelsize": 9, "xtick.color": INK2, "ytick.color": INK2,
        "xtick.labelsize": 8, "ytick.labelsize": 8, "axes.grid": True, "grid.color": GRIDC, "grid.linewidth": 0.5,
        "grid.linestyle": "-", "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False,
        "legend.fontsize": 8, "font.size": 9, "lines.linewidth": 1.6, "lines.solid_capstyle": "round",
        "font.family": "serif", "mathtext.fontset": "cm"})


def save_figure(fig, path, dpi=300):
    import matplotlib.pyplot as plt
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    return path

Writing clinic_core.py


In [3]:
# Copy the module to Drive (single source for the other notebooks) and import it
CORE = os.path.join(BASE, 'core', 'clinic_core.py')
shutil.copy('clinic_core.py', CORE)
sys.path.insert(0, os.getcwd())
import importlib
import clinic_core as cc
importlib.reload(cc)
P = cc.make_paths(BASE)
print('Core module written to', CORE)
print('numba available:', cc.HAVE_NUMBA)

Core module written to /content/drive/My Drive/clinic2026_final/core/clinic_core.py
numba available: True


In [4]:
# Compile the engine once and measure its speed, to plan the enumeration sessions of notebook 01
s = cc.setting(3, 1.6, 'moderate')
ev = cc.Evaluator(s, *cc.MAIN)
t0 = time.time()
ev.evaluate_rows(np.arange(20))
compile_seconds = time.time() - t0
n_bench = 2000 if cc.HAVE_NUMBA else 20
t0 = time.time()
ev.evaluate_rows(np.arange(20, 20 + n_bench))
rate = n_bench * cc.MAIN[0] / (time.time() - t0)
jobs = cc.enumeration_jobs()
day_sims = sum(cc.geometry(j[0])[2] * j[1] for j in jobs)
hours = day_sims / rate / 3600
pd.DataFrame([{'cpu cores': os.cpu_count(), 'compile seconds': compile_seconds, 'simulated days per second': rate,
               'enumeration jobs': len(jobs), 'simulated days in all jobs': day_sims,
               'estimated enumeration hours': hours}]).to_csv(os.path.join(P['results'], 'timing.csv'), index=False)
print(f'{os.cpu_count()} cores; first call (compilation) {compile_seconds:.0f} s; {rate:,.0f} simulated days per second')
print(f'Notebook 01 will simulate {day_sims / 1e9:.2f} billion days: about {hours:.1f} hours, '
      f'that is about {math.ceil(hours / 0.8)} sessions of 50 minutes')

2 cores; first call (compilation) 5 s; 91,165 simulated days per second
Notebook 01 will simulate 0.96 billion days: about 2.9 hours, that is about 4 sessions of 50 minutes


## 2. Unit tests

The tests check the finite-horizon logic that no closed form covers. Every hand-computed case runs through both the compiled engine and the plain reference engine, which must agree exactly.

- **Engine:**
  - hand-computed days;
  - the shared reception line;
  - staff leaving and arriving at block boundaries;
  - the admission cutoff;
  - conservation of work, capacity, first-come-first-served order and no idling, on random days;
  - wage, overtime and service-level accounting;
  - the arrival process;
  - a long-run comparison with Erlang C.
- **Analytic baselines:**
  - offered loads against closed forms and against a simulated infinite-server clinic;
  - the vectorized stationary rules against plan-by-plan evaluation;
  - the wait convolution;
  - MOL plan-level values against patient averages.
- **Analysis helpers:**
  - two-dimensional indicators;
  - minimal staffings and the product factor;
  - the descent;
  - the chunked enumeration, which must equal direct evaluation.

In [5]:
# Test definitions
ONE_BLOCK = [0.0, cc.T]


def run_both(arr, svc, staff, bounds=ONE_BLOCK, trace=None):
    arr = np.asarray(arr, dtype=float)
    svc = np.asarray(svc, dtype=float).reshape(-1, 5)
    staff = np.asarray(staff, dtype=np.int64)
    if staff.ndim == 1:
        staff = staff.reshape(4, 1)
    bounds = np.asarray(bounds, dtype=float)
    w, last, ho = cc.simulate_day_reference(arr, svc, staff, bounds, trace=trace)
    n = len(arr)
    a = np.full(n + 1, np.inf)
    a[:n] = arr
    s = np.zeros((n + 1, 5))
    s[:n] = svc
    w2, l2, h2 = np.empty(n + 1), np.empty(4), np.empty(4)
    cc._simulate_day(a, s, n, staff, bounds, w2, l2, h2)
    np.testing.assert_allclose(w2[:n], w, atol=1e-9)
    np.testing.assert_allclose(l2, last, atol=1e-9)
    np.testing.assert_allclose(h2, ho, atol=1e-9)
    return w, last, ho


def approx(a, b, rel=1e-9, abs_=0.0):
    return math.isclose(float(a), float(b), rel_tol=rel, abs_tol=abs_)


# ---------------------------------------------------------------- engine: hand-computable days
def test_single_patient_never_waits():
    w, last, ho = run_both([10.0], [[6, 10, 15, 8, 6]], [1, 1, 1, 1])
    assert w.tolist() == [0.0]
    assert last.tolist() == [55.0, 26.0, 41.0, 49.0]
    assert ho.tolist() == [0.0] * 4


def test_two_patients_deterministic_service():
    w, last, _ = run_both([0.0, 1.0], [[6, 10, 15, 8, 6]] * 2, [1, 1, 1, 1])
    assert w.tolist() == [0.0, 14.0]
    assert last.tolist() == [60.0, 26.0, 46.0, 54.0]


def test_checkout_shares_the_reception_line():
    w, last, _ = run_both([0.0, 5.0], [[1, 1, 1, 1, 5], [6, 1, 1, 1, 1]], [1, 1, 1, 1])
    assert w.tolist() == [0.0, 4.0]
    assert last[0] == 19.0


def test_reception_line_is_first_come_first_served_across_visits():
    w, last, _ = run_both([0.0, 0.5, 3.0], [[1, 1, 1, 1, 1], [10, 1.5, 1, 1, 1], [2, 1, 1, 1, 1]], [1, 1, 1, 1])
    assert w.tolist() == [9.0, 0.5, 8.0]
    assert last.tolist() == [17.0, 14.0, 15.0, 16.0]


def test_two_reception_visits_per_patient():
    rng = np.random.default_rng(3)
    arr = np.sort(rng.uniform(0, 400, 40))
    svc = rng.uniform(1, 12, (40, 5))
    trace = []
    run_both(arr, svc, [2, 2, 2, 2], trace=trace)
    per_patient = {}
    for j, k, r, *_ in trace:
        per_patient.setdefault(j, []).append((k, r))
    for j in range(40):
        assert sorted(per_patient[j]) == [(0, 0), (1, 1), (2, 2), (3, 3), (4, 0)]
    assert sum(r == 0 for _, _, r, *_ in trace) == 80


# ---------------------------------------------------------------- engine: shift changes
def test_shift_end_server_finishes_patient_then_leaves():
    bounds = [0.0, 100.0, cc.T]
    staff = [[2, 1], [3, 3], [3, 3], [3, 3]]
    w, last, ho = run_both([95.0, 97.0, 101.0], [[10, 1, 1, 1, 1], [20, 1, 1, 1, 1], [2, 1, 1, 1, 1]], staff, bounds)
    assert w.tolist() == [11.0, 0.0, 16.0]
    assert ho.tolist() == [5.0, 0.0, 0.0, 0.0]
    assert last[0] == 123.0


def test_capacity_increase_starts_waiting_patients_at_the_boundary():
    bounds = [0.0, 100.0, cc.T]
    staff = [[3, 3], [3, 3], [1, 2], [3, 3]]
    w, last, ho = run_both([0.0, 10.0], [[1, 1, 150, 1, 1], [1, 1, 10, 1, 1]], staff, bounds)
    assert w.tolist() == [0.0, 88.0]
    assert last[2] == 152.0
    assert ho.tolist() == [0.0] * 4


def test_no_handover_pay_when_staffing_rises_or_stays():
    rng = np.random.default_rng(5)
    arr = np.sort(rng.uniform(0, 470, 60))
    svc = rng.uniform(1, 15, (60, 5))
    _, _, ho = run_both(arr, svc, [[1, 2], [2, 2], [2, 3], [1, 1]], [0.0, 240.0, cc.T])
    assert ho.tolist() == [0.0] * 4


# ---------------------------------------------------------------- engine: admission cutoff and arrivals
def test_cutoff_edges():
    arr = np.array([100.0, 420.0, 449.99, 450.0, 479.9, np.inf, np.inf])
    assert cc.admitted_counts(arr, (420.0, 450.0, 480.0)).tolist() == [1, 3, 5]


def test_arrivals_stay_inside_the_opening_day():
    prof = cc.Profile(0.12, 1.6)
    E, _ = cc.draw_days(50)
    for i in range(50):
        a = prof.arrival_times(E[i])
        assert np.all(np.diff(a) > 0) and a.min() >= 0 and a.max() < cc.T


def test_evaluator_admits_only_before_cutoff():
    s = cc.Setting(cutoffs=(420.0, 450.0, 480.0))
    ev = cc.Evaluator(s, n_days=30)
    for i in range(30):
        for ci, c in enumerate(s.cutoffs):
            n = ev.n_adm[i, ci]
            assert np.all(ev.arr[i, :n] < c)
            assert n == ev.n_day[i] or ev.arr[i, n] >= c


# ---------------------------------------------------------------- engine: conservation and discipline
def test_work_conservation_capacity_fcfs_and_no_idling():
    for seed in range(12):
        rng = np.random.default_rng(100 + seed)
        B = int(rng.integers(1, 5))
        bounds = np.linspace(0.0, cc.T, B + 1)
        staff = rng.integers(1, 4, size=(4, B))
        n = int(rng.integers(20, 90))
        arr = np.sort(rng.uniform(0, 470, n))
        svc = rng.lognormal(np.log(cc.STAGE_MEANS), 0.6, (n, 5))
        trace = []
        w, last, ho = run_both(arr, svc, staff, bounds, trace=trace)
        tr = np.array(trace)
        assert len(tr) == 5 * n
        for r in range(4):
            rows = tr[tr[:, 2] == r]
            need = sum(svc[j, k] for j in range(n) for k in range(5) if cc.ROUTE[k] == r)
            assert math.isclose((rows[:, 5] - rows[:, 4]).sum(), need, rel_tol=1e-9)
            assert math.isclose(last[r], rows[:, 5].max(), rel_tol=1e-12)
        for j in range(n):
            rows = tr[tr[:, 0] == j]
            rows = rows[np.argsort(rows[:, 1])]
            assert rows[0, 3] == arr[j]
            assert np.allclose(rows[1:, 3], rows[:-1, 5])
            assert np.all(rows[:, 4] >= rows[:, 3] - 1e-12)
            assert math.isclose(w[j], (rows[:, 4] - rows[:, 3]).sum(), abs_tol=1e-9)
        for r in range(4):
            rows = tr[tr[:, 2] == r]
            for row in rows:
                t = row[4]
                b = min(int(np.searchsorted(bounds, t, side='right')) - 1, B - 1)
                assert np.sum((rows[:, 4] <= t) & (rows[:, 5] > t)) <= staff[r, b]
            order = np.lexsort((rows[:, 4], rows[:, 3]))
            assert np.all(np.diff(rows[order, 4]) >= -1e-9)
            ends = set(np.round(rows[:, 5], 9))
            for row in rows[rows[:, 4] > rows[:, 3] + 1e-12]:
                assert round(row[4], 9) in ends or round(row[4], 9) in set(np.round(bounds, 9))


def test_engines_agree_on_random_days():
    for seed in range(20):
        rng = np.random.default_rng(seed)
        B = int(rng.integers(1, 5))
        s = cc.Setting(n_blocks=B, par=float(rng.uniform(1.0, 2.0)), cv=(0.5, 1.0, 1.0, 0.5, 0.5))
        ev = cc.Evaluator(s, n_days=3, seed_offset=1000 * seed)
        staff = rng.integers(1, 4, size=(4, B))
        for i in range(3):
            n = ev.n_adm[i, 0]
            run_both(ev.arr[i, :n], ev.svc[i, :n], staff, s.bounds)


# ---------------------------------------------------------------- engine: costs and service level
def test_overtime_handover_and_wage_accounting():
    bounds = np.array([0.0, 240.0, cc.T])
    staff = np.array([[1, 2], [1, 1], [2, 1], [1, 3]], dtype=np.int64)
    arr = np.array([470.0, np.inf])
    svc = np.zeros((2, 5))
    svc[0] = [6, 10, 15, 8, 6]
    D = np.empty((1, 15))
    cc._plan_days(arr[None, :], svc[None, :, :], np.array([1]), staff, bounds, D)
    out = np.empty(len(cc.METRIC_COLS))
    cc._metrics_from_days(D, staff, bounds, cc.WAGES, cc.KAPPA, cc.C_W, cc.T, cc.TAU, out)
    m = dict(zip(cc.METRIC_COLS, out))
    overtime = np.array([515.0, 486.0, 501.0, 509.0]) - cc.T
    wages = float(sum(cc.WAGES[r] / cc.T * 240 * staff[r].sum() for r in range(4)))
    ot_pay = cc.KAPPA * float((cc.WAGES / cc.T * staff[:, 1]) @ overtime)
    assert approx(m['wages'], wages) and approx(m['OT_pay'], ot_pay) and approx(m['M'], wages + ot_pay)
    assert m['O'] == 35.0 and m['W'] == 0.0 and m['SL'] == 1.0 and m['handover_pay'] == 0.0


def test_handover_minutes_are_paid_at_the_overtime_rate():
    bounds = np.array([0.0, 100.0, cc.T])
    staff = np.array([[2, 1], [3, 3], [3, 3], [3, 3]], dtype=np.int64)
    arr = np.array([[95.0, 97.0, 101.0, np.inf]])
    svc = np.zeros((1, 4, 5))
    svc[0, :3] = [[10, 1, 1, 1, 1], [20, 1, 1, 1, 1], [2, 1, 1, 1, 1]]
    D = np.empty((1, 15))
    cc._plan_days(arr, svc, np.array([3]), staff, bounds, D)
    out = np.empty(len(cc.METRIC_COLS))
    cc._metrics_from_days(D, staff, bounds, cc.WAGES, cc.KAPPA, cc.C_W, cc.T, cc.TAU, out)
    m = dict(zip(cc.METRIC_COLS, out))
    assert approx(m['handover_pay'], cc.KAPPA * cc.WAGES[0] / cc.T * 5.0)
    assert approx(m['W'], cc.C_W * 27.0) and approx(m['SL'], 1.0)


def test_service_level_convention_on_a_day_without_admissions():
    staff = np.ones((4, 1), dtype=np.int64)
    D = np.empty((1, 15))
    cc._plan_days(np.full((1, 2), np.inf), np.zeros((1, 2, 5)), np.array([0]), staff, np.array([0.0, cc.T]), D)
    out = np.empty(len(cc.METRIC_COLS))
    cc._metrics_from_days(D, staff, np.array([0.0, cc.T]), cc.WAGES, cc.KAPPA, cc.C_W, cc.T, cc.TAU, out)
    m = dict(zip(cc.METRIC_COLS, out))
    assert m['SL'] == 1.0 and m['W'] == 0.0 and m['OT_pay'] == 0.0 and m['M'] == cc.WAGES.sum()


def test_enumeration_rows_equal_cached_evaluation():
    s = cc.setting(2, 1.6, 'high')
    ev = cc.Evaluator(s, n_days=5, seed_offset=77)
    rows = np.array([0, 17, 1234, 6560])
    A = ev.evaluate_rows(rows)
    B = ev.evaluate_many([cc.plan_of_row(r, s) for r in rows])
    assert np.array_equal(A, B)


# ---------------------------------------------------------------- arrival process
def test_profile_mean_and_peak():
    for par in cc.PARS:
        prof = cc.Profile(0.12, par)
        assert approx(np.trapezoid(prof.g, prof.t) / cc.T, 1.0)
        assert approx(prof.g.max(), par)
        assert approx(prof.Lambda(cc.T), 0.12 * cc.T)


def test_nonhomogeneous_poisson_counts():
    prof = cc.Profile(0.12, 1.6)
    E, _ = cc.draw_days(3000, seed_offset=50_000)
    counts = np.array([[np.sum(prof.arrival_times(E[i]) < c) for c in (120.0, 240.0, 480.0)] for i in range(3000)])
    for k, c in enumerate((120.0, 240.0, 480.0)):
        lam = prof.Lambda(c)
        assert abs(counts[:, k].mean() - lam) < 4 * math.sqrt(lam / 3000)
        assert abs(counts[:, k].var() / lam - 1) < 0.1


def test_long_run_waits_match_erlang_c():
    lam, staff = 0.12, np.array([3, 2, 3, 2], dtype=np.int64)
    horizon, warm = 120_000.0, 10_000.0

    def mmc_wait(L, m, c):
        a = L * m
        tail = a ** c / math.factorial(c) / (1 - a / c)
        pw = tail / (sum(a ** k / math.factorial(k) for k in range(c)) + tail)
        return pw / (c / m - L)

    theory = 2 * mmc_wait(2 * lam, 6.0, 3) + mmc_wait(lam, 10.0, 2) + mmc_wait(lam, 15.0, 3) + mmc_wait(lam, 8.0, 2)
    reps = []
    for rep in range(8):
        rng = np.random.default_rng(9000 + rep)
        arr = np.cumsum(rng.exponential(1 / lam, int(horizon * lam * 1.3)))
        arr = arr[arr < horizon]
        n = len(arr)
        svc = rng.exponential(1.0, (n, 5)) * cc.STAGE_MEANS
        w = np.empty(n + 1)
        cc._simulate_day(np.append(arr, np.inf), np.vstack([svc, np.zeros((1, 5))]), n, staff.reshape(4, 1),
                         np.array([0.0, horizon]), w, np.empty(4), np.empty(4))
        keep = (arr > warm) & (arr < horizon - warm)
        reps.append(w[:n][keep].mean())
    reps = np.array(reps)
    half = 2.36 * reps.std(ddof=1) / math.sqrt(len(reps))
    assert abs(reps.mean() - theory) < max(half, 0.03 * theory)


# ---------------------------------------------------------------- analytic baselines: offered loads
def exponential_H(mean):
    return lambda y: np.where(y > 0, y - mean * (1 - np.exp(-np.maximum(y, 0.0) / mean)), 0.0)


def test_constant_rate_long_after_opening_gives_lambda_times_mean():
    for mean, cv in ((6.0, 0.5), (15.0, 1.0), (10.0, 0.2)):
        lam, horizon = 0.12, 720
        loads, _ = cc.offered_loads_from_input(np.full(horizon, lam), [cc.stage_kernels(mean, cv, horizon)], horizon)
        assert approx(loads[0, -1], lam * mean, rel=1e-5)


def test_exponential_service_with_step_input():
    lam, mean, horizon = 0.1, 10.0, 300
    kern = [cc.stage_kernels(mean, 1.0, horizon, H=exponential_H(mean))]
    loads, _ = cc.offered_loads_from_input(np.full(horizon, lam), kern, horizon)
    t = np.arange(horizon + 1)
    np.testing.assert_allclose(loads[0], lam * mean * (1 - np.exp(-t / mean)), atol=1e-12)


def test_kernels_are_probability_distributions():
    G, q = cc.stage_kernels(15.0, 1.0, 720)
    assert q.min() >= -1e-15 and abs(q.sum() - 1.0) < 1e-6
    assert approx(G.sum(), 15.0, rel=1e-4)


def test_tandem_offered_loads_match_infinite_server_simulation():
    s = cc.Setting(n_blocks=2, mean_rate=0.10, par=2.0, cv=(0.5, 1.0, 1.0, 0.5, 0.5))
    c = 450.0
    loads = cc.offered_loads(s, c)
    prof = s.profile()
    rng = np.random.default_rng(2024)
    n_days = 20_000
    E = np.cumsum(rng.exponential(1.0, (n_days, 120)), axis=1)
    arr = np.interp(E, prof.cum, prof.t)
    arr = np.where(E < prof.Lambda(c), arr, np.nan)
    sig = np.sqrt(np.log1p(np.asarray(s.cv) ** 2))
    svc = cc.STAGE_MEANS * np.exp(sig * rng.standard_normal((n_days, 120, 5)) - sig ** 2 / 2)
    start = arr[..., None] + np.concatenate([np.zeros((n_days, 120, 1)), np.cumsum(svc, axis=2)[..., :4]], axis=2)
    for k in range(5):
        for t in np.arange(30, 600, 30):
            inside = (start[..., k] <= t) & (t < start[..., k] + svc[..., k])
            counts = np.nansum(inside, axis=1)
            m, se = counts.mean(), counts.std(ddof=1) / math.sqrt(n_days)
            assert abs(m - loads[k, t]) <= 4.0 * se + 0.01, (k, t, m, loads[k, t])


def test_reception_load_adds_checkin_and_checkout():
    s = cc.Setting(n_blocks=2, mean_rate=0.10, par=1.6, cv=(0.5,) * 5)
    mol = cc.MOL(s, 480.0)
    loads = cc.offered_loads(s, 480.0)
    np.testing.assert_allclose(mol.role_load[0], loads[0] + loads[4])


# ---------------------------------------------------------------- analytic baselines: screens and convolution
def test_screens_reproduce_period_by_period_tables():
    s = cc.Setting(n_blocks=2, mean_rate=0.10, par=1.6, cv=(0.5,) * 5, cutoffs=(420.0, 450.0, 480.0))
    scr = cc.Screens(s, mol=False)
    D = cc.staffing_digits(s, np.arange(3 ** 8))
    for c in s.cutoffs:
        plans = np.column_stack([D + 1, np.full(len(D), c)]).astype(float)
        res = scr.evaluate(D, c, variants=('avg-block', 'avg-daily', 'lag-block', 'lag-daily', 'max-block',
                                           'max-daily'))
        for k in ('avg', 'lag', 'max'):
            t = cc.sipp_table(s, k, plans=plans)
            stb = t.stable.values
            for rule, col in (('block', 'SL_block_hat'), ('daily', 'SL_daily_hat')):
                r = res[f'{k}-{rule}']
                assert np.array_equal(r['feas'], stb & (t[col].values >= cc.TAU))
                np.testing.assert_allclose(r['SL'], np.where(stb, t[col].values, 0.0))
                np.testing.assert_allclose(r['M'][stb], t.M_hat.values[stb])
                np.testing.assert_allclose(r['W'][stb], t.W_hat.values[stb])


def test_path_service_levels_match_direct_convolution():
    rng = np.random.default_rng(5)
    stable = rng.random((5, 3)) > 0.15
    pw = rng.uniform(0.0, 0.9, (5, 3))
    cond = rng.uniform(0.3, 25.0, (5, 3))
    spec, n = cc.stage_spectra(stable, pw, cond)
    Pv = cc.path_service_levels(spec, n)
    for combo in range(243):
        lv = [(combo // 3 ** (4 - k)) % 3 for k in range(5)]
        ref = cc.prob_total_wait_le([(pw[k, lv[k]], cond[k, lv[k]]) for k in range(5)]) \
            if all(stable[k, lv[k]] for k in range(5)) else 0.0
        assert abs(Pv[combo] - ref) < 1e-9


def test_station_arrays_match_station_model():
    s = cc.Setting(n_blocks=2, mean_rate=0.10, par=1.0, cv=(0.5, 1.0, 1.0, 0.5, 0.5))
    for role in range(4):
        es, cs2, visits = cc.station_moments(s, role)
        for servers in (1, 2, 3):
            for lam in (0.02, 0.05, 0.09):
                ref = cc.station_model(lam, s, role, servers)
                stb, pw, wq, cond = cc.station_arrays(lam * visits * es, servers, es, cs2)
                if ref is None:
                    assert not stb
                else:
                    assert stb and approx(pw, ref[0], rel=1e-12) and approx(wq, ref[1], rel=1e-12)


def test_mol_plan_level_equals_patient_average():
    s = cc.Setting(n_blocks=3, mean_rate=0.10, par=2.0, cv=(0.5,) * 5)
    scr = cc.Screens(s, kinds=())
    rng = np.random.default_rng(1)
    D = rng.integers(0, 3, (50, 12))
    r = scr.evaluate(D, 480.0, variants=('mol-daily',))['mol-daily']
    mol = scr.mol[480.0]
    for i in range(0, 50, 7):
        stf = (D[i] + 1).reshape(4, 3)
        assert approx(r['SL'][i], (mol.adm * mol.patient_levels(stf)).sum() / mol.adm.sum(), rel=1e-12)
        assert approx(r['W'][i], cc.C_W * (mol.adm * mol.patient_waits(stf)).sum(), rel=1e-9)


# ---------------------------------------------------------------- analysis helpers
def test_hypervolume_2d_on_hand_computed_points():
    pts = np.array([[0.0, 1.0], [0.5, 0.5], [1.0, 0.0], [0.6, 0.6], [1.2, 0.0]])
    assert approx(cc.hv2(pts, (1.1, 1.1)), 0.11 + 0.30 + 0.05)
    assert approx(cc.hv2(np.array([[0.2, 0.3]]), (1.0, 1.0)), 0.8 * 0.7)
    assert cc.hv2(np.zeros((0, 2)), (1.1, 1.1)) == 0.0


def test_pareto2():
    M = np.array([3.0, 1.0, 2.0, 2.0, 4.0])
    W = np.array([1.0, 3.0, 2.0, 2.5, 0.5])
    assert cc.pareto2(M, W).tolist() == [0, 1, 2, 4]


def synthetic_table(s, sl_fn):
    nv, L, n_stf, pw = cc.geometry(s)
    X = cc.staffing_digits(s, np.arange(n_stf)) + 1
    df = pd.DataFrame(X, columns=cc.staffing_cols(s))
    df['Cutoff'] = float(s.cutoffs[0])
    df['SL'] = [sl_fn(x) for x in X]
    blen = np.diff(s.bounds)
    df['M'] = [float(sum(cc.WAGES[r] / cc.T * blen[b] * x[r * s.n_blocks + b] for r in range(4)
                         for b in range(s.n_blocks))) for x in X]
    df['W'] = 0.0
    df['feasible'] = df.SL >= cc.TAU
    return df


def test_minimal_staffings_and_product_factor():
    s = cc.Setting(n_blocks=2)
    # per-block condition: at least 2 staff of every role in every block -> a box, product factor 1
    df = synthetic_table(s, lambda x: 0.95 if x.min() >= 2 else 0.5)
    assert len(cc.minimal_staffings(df, s)) == 1 and approx(cc.product_factor(df, s), 1.0)
    # cross-block compensation: physicians over the two blocks at least 4 -> several minimal staffings, factor > 1
    df = synthetic_table(s, lambda x: 0.95 if x[4] + x[5] >= 4 else 0.5)
    mins = cc.minimal_staffings(df, s)
    assert len(mins) == 3 and cc.product_factor(df, s) > 1.0


def test_descent_finds_cheapest_and_uses_swaps():
    s = cc.Setting(n_blocks=1)
    df = synthetic_table(s, lambda x: 0.95 if (x[0] >= 2 and x[1] + x[2] >= 4) else 0.5)
    o = cc.TableOracle(df, s)
    assert cc.first_improvement(o, s, (3, 3, 3, 3, 480.0), np.random.default_rng(0)) == (2, 3, 1, 1, 480.0)
    o = cc.TableOracle(df, s)       # from (2, 2, 2, 1) only replacing a physician by a nurse improves
    assert cc.first_improvement(o, s, (2, 2, 2, 1, 480.0), np.random.default_rng(0)) == (2, 3, 1, 1, 480.0)


def test_chunked_enumeration_queue_equals_direct_evaluation():
    import tempfile
    s = cc.setting(2, 1.3, 'high', levels=2)
    with tempfile.TemporaryDirectory() as tmp:
        cc.run_enumeration_queue(tmp, [(s, 4, 500)], max_minutes=10, target_day_sims=150, verbose=False)
        df = cc.load_table(tmp, s, 4, 500)
    ev = cc.Evaluator(s, 4, 500)
    X = ev.evaluate_many([cc.plan_of_row(r, s) for r in range(len(df))])
    idx = [cc.METRIC_COLS.index(k) for k in cc.STORED]
    assert np.array_equal(df[list(cc.STORED)].values, X[:, idx])


TESTS = [(name, fn) for name, fn in list(globals().items()) if name.startswith('test_') and callable(fn)]
print(len(TESTS), 'tests defined')

34 tests defined


In [6]:
# Run the tests (about 2 minutes)
results = []
for name, fn in TESTS:
    t0 = time.time()
    try:
        fn()
        ok, msg = True, ''
    except Exception as e:
        ok, msg = False, f'{type(e).__name__}: {e}'
    results.append({'test': name[5:].replace('_', ' '), 'passed': ok, 'seconds': round(time.time() - t0, 1),
                    'message': msg})
    print(('PASS ' if ok else 'FAIL ') + name, msg)
tests = pd.DataFrame(results)
tests.to_csv(os.path.join(P['results'], 'unit_tests.csv'), index=False)
print(f'\n{int(tests.passed.sum())} of {len(tests)} tests passed')
assert tests.passed.all(), 'Some tests failed: do not run the other notebooks until this is fixed'

PASS test_single_patient_never_waits 
PASS test_two_patients_deterministic_service 
PASS test_checkout_shares_the_reception_line 
PASS test_reception_line_is_first_come_first_served_across_visits 
PASS test_two_reception_visits_per_patient 
PASS test_shift_end_server_finishes_patient_then_leaves 
PASS test_capacity_increase_starts_waiting_patients_at_the_boundary 
PASS test_no_handover_pay_when_staffing_rises_or_stays 
PASS test_cutoff_edges 
PASS test_arrivals_stay_inside_the_opening_day 
PASS test_evaluator_admits_only_before_cutoff 
PASS test_work_conservation_capacity_fcfs_and_no_idling 
PASS test_engines_agree_on_random_days 
PASS test_overtime_handover_and_wage_accounting 
PASS test_handover_minutes_are_paid_at_the_overtime_rate 
PASS test_service_level_convention_on_a_day_without_admissions 
PASS test_enumeration_rows_equal_cached_evaluation 
PASS test_profile_mean_and_peak 
PASS test_nonhomogeneous_poisson_counts 
PASS test_long_run_waits_match_erlang_c 
PASS test_constant_rate

In [7]:
# Helpers that write the tables in the exact format of the paper
EPS = 1e-9


def order(d):
    return d.assign(_cv=d.cv.map({'moderate': 0, 'high': 1})).sort_values(['blocks', 'PAR', '_cv'])


def table_tex(colspec, head, body, cap, label, box=r'max width=\textwidth, max totalheight=0.88\textheight'):
    """A table in adjustbox with a pipe-delimited tabular, as in the manuscript."""
    return '\n'.join([r'\begin{table}[htbp]', r'\centering', rf'\begin{{adjustbox}}{{{box}}}',
                      rf'\begin{{tabular}}{{{colspec}}}', r'\hline', head, r'\hline'] + list(body)
                     + [r'\end{tabular}', r'\end{adjustbox}', rf'\caption{{{cap}}}', rf'\label{{{label}}}',
                        r'\end{table}']) + '\n'


def write_table(name, text):
    path = os.path.join(P['tables'], name)
    cc.write_text(path, text)
    print('Written', path)

## 3. Engine against Erlang C

The check uses exponential service times, one block, a constant arrival rate of 0.10 per minute and a clinic that never closes. Under those conditions the clinic is an open network of multi-server first-come-first-served stations with a product-form stationary distribution, so the mean wait per visit at each station must equal the Erlang C value at the station's total visit rate.

The check makes 30 independent runs of 200,000 minutes for each of two staffings. It uses the plain Python reference engine because that engine records the wait of every visit. The compiled engine agrees with the reference engine exactly, which the unit tests confirm.

In [8]:
# Erlang C check and Table B.1 (the check is read from its CSV when it exists, unless RERUN_ERLANG_C is True)
RERUN_ERLANG_C = False
path = os.path.join(P['results'], 'verification_erlang_c.csv')
if os.path.exists(path) and not RERUN_ERLANG_C:
    ver = pd.read_csv(path)
    print('Read', path)
else:
    t0 = time.time()
    ver = cc.erlang_c_check(n_runs=30)
    ver.to_csv(path, index=False)
    print(f'Simulated in {time.time() - t0:.0f} s')
print(ver.round(3).to_string(index=False))
print(f'Every Erlang C value inside the 95% interval: {bool(ver["inside CI"].all())}; relative errors '
      f'{ver["relative error %"].min():+.1f}% to {ver["relative error %"].max():+.1f}%')

body = []
for staff, g in ver.groupby('staffing', sort=False):
    for _, r in g.iterrows():
        body.append(f"{staff} & {r.station} & {int(r.servers)} & {r.utilization:.2f} & "
                    f"{r.simulated:.2f} [{r['CI low']:.2f}, {r['CI high']:.2f}] & {r['Erlang C']:.2f} & "
                    f"{r['relative error %']:+.1f} \\\\")
    body.append(r'\hline')
head = (r'Staffing & Station & Servers & Utilization & Simulated wait (min), 95\% CI & Erlang C (min) & '
        r'Relative error (\%) \\')
cap = (r'Mean wait per visit in minutes, simulated and by Erlang C, with exponential service times, one block and a '
       r'constant arrival rate of 0.10 per minute. Staffing lists receptionists, nurses, physicians and pharmacists.')
write_table('verification.tex', table_tex('|l|l|c|c|c|c|c|', head, body, cap, 'tab:app_verification'))

Simulated in 29 s
    staffing              station  servers  utilization  simulated  CI low  CI high  Erlang C  relative error %  inside CI
(2, 2, 2, 1) Reception (check-in)        2         0.60      3.358   3.279    3.437     3.375            -0.505       True
(2, 2, 2, 1)                Nurse        2         0.50      3.331   3.268    3.393     3.333            -0.083       True
(2, 2, 2, 1)            Physician        2         0.75     19.028  18.448   19.607    19.286            -1.338       True
(2, 2, 2, 1)             Pharmacy        1         0.80     32.014  30.711   33.317    32.000             0.044       True
(2, 2, 2, 1) Reception (checkout)        2         0.60      3.351   3.273    3.430     3.375            -0.702       True
(3, 2, 3, 2) Reception (check-in)        3         0.40      0.470   0.459    0.481     0.471            -0.118       True
(3, 2, 3, 2)                Nurse        2         0.50      3.325   3.263    3.388     3.333            -0.240       Tru

## 4. Reproduction of the earlier numbers

The engine, the random numbers and the analytic baselines are those of the earlier version of the study, restricted to admission until closing. The plans below were simulated in that version. Their labor cost, waiting cost and service level on the 200-day main sample must be reproduced to rounding error.

In [9]:
REFERENCE_VALUES = [   # setting, staffing (Rec b1..bB, Nur.., Doc.., Pha..), M, W, SL from the earlier version
    ('B2-PAR1.0-moderate', (3, 2, 2, 2, 2, 3, 1, 2), 1552.5484848393874, 93.12364768193083, 0.904010119412036),
    ('B2-PAR1.0-moderate', (3, 3, 3, 3, 3, 3, 3, 3), 2106.938689300159, 17.628258554307003, 0.9979665964059208),
    ('B2-PAR2.0-high', (3, 3, 3, 2, 3, 3, 2, 2), 1845.747599498578, 76.02315238076967, 0.918183493694838),
    ('B3-PAR1.6-moderate', (3, 2, 3, 2, 1, 2, 3, 2, 3, 2, 1, 2), 1555.2222050446144, 89.2977132785721,
     0.9057194553758832),
    ('B3-PAR2.0-high', (3, 2, 3, 3, 2, 2, 3, 2, 3, 3, 2, 2), 1745.418196770764, 88.97721678475384,
     0.9000769125008621),
]
settings = {s.name: s for s in cc.grid()}
worst = 0.0
for name, staff, M, W, SL in REFERENCE_VALUES:
    s = settings[name]
    x = cc.Evaluator(s, *cc.MAIN).evaluate_many([staff + (cc.CUTOFF,)])[0]
    got = np.array([x[cc.METRIC_COLS.index(k)] for k in ('M', 'W', 'SL')])
    diff = float(np.abs(got - np.array([M, W, SL])).max())
    worst = max(worst, diff)
    print(f'{name} {cc.staffing_text(staff, s)}: M {got[0]:.4f}, W {got[1]:.4f}, SL {got[2]:.6f}; '
          f'largest difference {diff:.1e}')
assert worst < 1e-8, 'The engine does not reproduce the earlier numbers'
print('Earlier numbers reproduced.')

B2-PAR1.0-moderate (3,2,2,1 | 2,2,3,2): M 1552.5485, W 93.1236, SL 0.904010; largest difference 1.1e-16
B2-PAR1.0-moderate (3,3,3,3 | 3,3,3,3): M 2106.9387, W 17.6283, SL 0.997967; largest difference 1.1e-16
B2-PAR2.0-high (3,3,3,2 | 3,2,3,2): M 1845.7476, W 76.0232, SL 0.918183; largest difference 1.1e-16
B3-PAR1.6-moderate (3,2,3,2 | 2,1,2,1 | 3,2,3,2): M 1555.2222, W 89.2977, SL 0.905719; largest difference 0.0e+00
B3-PAR2.0-high (3,3,3,3 | 2,2,2,2 | 3,2,3,2): M 1745.4182, W 88.9772, SL 0.900077; largest difference 2.3e-13
Earlier numbers reproduced.


## 5. Parameter table

The next cell writes Table 2. The within-block peakedness and the number of plans are computed from the settings, so the table cannot drift from the code.

In [10]:
# Table 2: parameters and factor levels of the 16 settings
eta = {B: [cc.peakedness(cc.setting(B, par, 'moderate')) for par in cc.PARS] for B in (2, 3)}
n_plans = {B: cc.geometry(cc.setting(B, 1.0, 'moderate'))[2] for B in (2, 3)}


def listing(xs):
    return ', '.join(f'{x:.2f}' for x in xs[:-1]) + f' and {xs[-1]:.2f}'


body = [
    r'Opening day & $T=480$ minutes. Walk-ins arriving in $[0,T)$ are admitted, and every admitted patient is served \\',
    r'Staffing blocks & $B=2$ blocks of 240 minutes or $B=3$ blocks of 160 minutes \\',
    rf'Staff per role and block & 1, 2 or 3 ($L=3$), which gives {n_plans[2]:,} plans with two blocks and {n_plans[3]:,} with three \\',
    r'Arrivals & Non-homogeneous Poisson, mean rate $\bar\lambda=0.10$ per minute (48 walk-ins a day), profile '
    r'\eqref{eq:profile} with peaks at $p_1=90$ and $p_2=360$ minutes, heights $a_1=1$ and $a_2=0.75$, $\sigma=50$ '
    r'minutes, and $\pi\in\{1.0,1.3,1.6,2.0\}$ \\',
    rf'Within-block peakedness $\eta$ & {listing(eta[2])} with two blocks, and {listing(eta[3])} with three, for the four values of $\pi$ \\',
    r'Service times & Log-normal, with means of 6 (each reception visit), 10, 15 and 8 minutes. Coefficient of variation '
    r'0.5 at every stage (moderate), or 1.0 at the nurse and physician stages and 0.5 elsewhere (high) \\',
    r'Wages and overtime & \$110, \$160, \$250 and \$130 per 480 minutes for receptionist, nurse, physician and '
    r'pharmacist, paid pro rata by block. Overtime and the minutes in $H_r$ are paid at $\kappa=1.5$ times the regular rate \\',
    r'Waiting cost & $c_W=\$0.167$ per patient-minute, half the per-minute wage of a nurse \\',
    r'Service standard & At least $\tau=0.90$ of the admitted patients wait at most $\omega=30$ minutes in total, in '
    r'expectation over days \\',
    r'Samples & Main sample of $n=200$ days. Reference sample of 1,000 days and, for the two-block settings, four further '
    r'independent samples of 200 days \\',
    r'Descent & Ten runs from the MOL pick and ten from full staffing in each setting in which the MOL accepts a plan, '
    r'each with its own random order of moves \\',
    r'\hline']
write_table('params.tex', table_tex('|l|p{11.6cm}|', r'Parameter & Value \\', body,
                                    'Parameters and factor levels of the 16 settings.', 'tab:params',
                                    box=r'max width=\textwidth'))

Written /content/drive/My Drive/clinic2026_final/tables/params.tex


**Next:** notebook 01, the enumeration. If its 56 jobs are already finished on Drive, do not run it again; go to notebook 02.